# SmolBench family-ladder study -- statistical analyses

One notebook for every statistic the study reports. It **imports the live
analysis modules** and calls them; it re-implements nothing that already has
a home in `notebooks/*/`. Sections 7-9 additionally implement three
statistics inline; everything else calls a live module.

## Rules this notebook runs under

1. **Archived data is accessed on S3, never written to a local path.**
   Sections 0, 5, 8 and 9 stream objects out of
   `s3://smolbench-results-414266451290/archives/2026-08-25/` straight into
   memory (`S3Archive` below, the same read/sha256 logic as
   `tests/conftest.py::S3Archive`). Nothing here downloads the archive. This
   is a different prefix and a different rule from what the heavy
   deduction cells in sections 5 and 6 do: they fetch RESULTS-STORE rows
   -- this study's live spool prefix, not the archive -- through
   `rows_source.resolve_rows_dir`, into a temporary directory that is
   scratch, not a checked-in path.
2. **`RUN_HEAVY` gates everything that needs the full results store**
   (`harness.sync_down()`, GPU or Lean work, and the S3 row fetches the
   heavy deduction cells in sections 5 and 6 make). Those cells are
   complete, runnable code -- they are switched off, not stubbed out.
3. **Saved with all outputs cleared.** Re-run it to reproduce the numbers.

Live AWS credentials are a *baseline* requirement: the ungated cells in
sections 0, 5, 8 and 9 read the archive. Everything else runs offline (or
skips).

## Section 0 -- setup and provenance

**What this is.** The anchor for everything below: the repository root, the
live analysis modules bound under unambiguous names, the archive handle, and
a live provenance listing (key + size + sha256) of every archived object this
notebook reads.

**Inputs.** The repo working tree (for the modules) and the S3 archive prefix
(for the evidence).

**Why `_load`/`_bound`.** Both legs ship a file called `power_analysis.py`,
and each module imports its siblings *by bare name* after putting its own
directory on `sys.path`; whichever leg imported first would otherwise own
`sys.modules["power_analysis"]` for the rest of the session. `_bound` binds
the right sibling for exactly the duration of each exec.

In [ ]:
"""Anchor the repo, then load every live analysis module under a unique name."""
import importlib.util
import json
import sys
from contextlib import contextmanager
from pathlib import Path


def find_repo(start: Path | None = None) -> Path:
    """Walk up from `start` (default: cwd) to the directory holding pyproject.toml.

    Notebooks have no ``__file__``, so the repo root is recovered from the
    working directory instead. This raises rather than guessing: a wrong
    root would silently point every path below at the wrong tree.
    """
    here = (start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / "pyproject.toml").is_file():
            return cand
    raise RuntimeError(
        f"no pyproject.toml at or above {here}: run this notebook from inside "
        "the SmolBench checkout"
    )


REPO = find_repo()
IND = REPO / "notebooks" / "induction" / "analysis"
DED = REPO / "notebooks" / "deduction" / "analysis"
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))


def _load(name: str, path: Path):
    """Exec the module at `path` under `name`, registering it before exec."""
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module          # dataclass annotations resolve via sys.modules
    spec.loader.exec_module(module)
    return module


@contextmanager
def _bound(**modules):
    """Bind modules under bare names for the duration of the block."""
    saved = {n: sys.modules.get(n) for n in modules}
    sys.modules.update(modules)
    try:
        yield
    finally:
        for n, old in saved.items():
            if old is None:
                sys.modules.pop(n, None)
            else:
                sys.modules[n] = old


# --- deduction leg ---------------------------------------------------------
# rows_source loads FIRST and stays bound under the bare name `rows_source`
# for as long as any of its three siblings might exec: power_analysis.py,
# error_bars.py AND hint_vs_noise.py all import it by bare name off their
# own sys.path insert. Loading it only after them would leave this notebook
# holding a SECOND `rows_source` module object -- with its own S3_BUCKET --
# while the scripts it goes on to call kept using the first. The notebook
# name `rows_source` below is bound to the SAME object the scripts call,
# which is the whole point: a report built from ROWS_DIR/RECOVERY_DIR in
# section 5 has to be the tree those scripts would also resolve.
rows_source = _load("ded_rows_source", DED / "rows_source.py")
with _bound(rows_source=rows_source):
    ded_pa = _load("ded_power_analysis", DED / "power_analysis.py")
    with _bound(power_analysis=ded_pa):
        error_bars = _load("ded_error_bars", DED / "error_bars.py")
        with _bound(error_bars=error_bars):
            hint_vs_noise = _load("ded_hint_vs_noise", DED / "hint_vs_noise.py")

# --- induction leg ---------------------------------------------------------
ind_pa = _load("ind_power_analysis", IND / "power_analysis.py")
with _bound(power_analysis=ind_pa):
    paired = _load("ind_paired_analysis", IND / "paired_analysis.py")
    with _bound(paired_analysis=paired):
        significance = _load("ind_significance_report", IND / "significance_report.py")
        with _bound(significance_report=significance):
            extens_vs_noise = _load("ind_extens_vs_noise", IND / "extens_vs_noise.py")
run_study = _load("ind_run_study", REPO / "notebooks" / "induction" / "run_study.py")

# ``notebooks/_power_common.py`` is imported by BOTH legs' power_analysis (each
# adds ``notebooks/`` to sys.path itself), so it is already in sys.modules;
# name it here so this notebook can cite its constants directly.
power_common = sys.modules["_power_common"]

print("repo          :", REPO)
print("interpreter   :", sys.executable)
print("induction lanes:", len(ind_pa.MODELS), " families:", len(ind_pa.FAMILIES))
print("deduction lanes:", len(ded_pa.MODELS), " families:", len(ded_pa.FAMILIES))
print("run_study roster:", len(run_study.MODELS), "lanes x", len(run_study.INFO_TYPES),
      "info arms, R =", run_study.N_REPLICATES)

In [ ]:
"""The heavy-work gate. Flip to True only with the results store in reach."""
#: Everything needing the FULL results store is behind this flag:
#: `harness.sync_down()` into `notebooks/induction/results`, GPU or Lean
#: work, and -- the S3 row fetches sections 5 and 6 make. The reader those
#: sections call (`rows_source.resolve_rows_dir`) owns its own
#: destination: it lands the fetched rows in a fresh temporary directory
#: and prints that path to stderr, never into the repo tree. What it
#: fetches is results-STORE data -- this study's live spool prefix -- not
#: the 2026-08-25 ARCHIVE; rule 1 above is about the archive and is
#: unaffected by this flag. The archive-backed cells (sections 0, 5's
#: recovery summary, 8, 9) stay UNGATED: they stream a few small JSON
#: objects and are the point of this notebook.
RUN_HEAVY = False

print(f"RUN_HEAVY = {RUN_HEAVY}")

In [ ]:
"""Read-only, in-memory access to the 2026-08-25 archive prefix on S3.

Same read/sha256 logic as ``tests/conftest.py::S3Archive`` (still copied, not
imported: ``tests/`` is not an importable package). The two primitives this
class used to re-implement are imported instead, from ``smolbench``, which IS
importable: the S3 client from ``smolbench.evals._aws.fresh_client`` and the
URI split from ``smolbench.evals.results_store.parse_s3_uri``. Nothing is
written to disk -- the user's ruling is that archived data is accessed on AWS.
"""
import hashlib
import posixpath

from smolbench.evals import _aws
from smolbench.evals.results_store import parse_s3_uri

ARCHIVE = "s3://smolbench-results-414266451290/archives/2026-08-25"
ARCHIVE_REGION = "us-west-2"


class S3Archive:
    """Stream objects out of an ``s3://bucket/prefix`` archive root.

    Parameters
    ----------
    uri : str
        ``s3://<bucket>/<prefix>`` of the archive root.
    region : str or None
        Region for the S3 client; ``None`` lets the SDK resolve one.

    Notes
    -----
    The client comes from ``_aws.fresh_client``, deliberately NOT the SDK's
    module-level ``client(...)`` helper: that helper goes through the
    process-wide default session, which resolves ``~/.aws/credentials`` ONCE
    and never re-reads it. A notebook kernel is exactly the long-lived process
    ``fresh_client``'s own docstring names -- it outlives the repo's
    short-lived IdP sessions -- so a cached session would keep signing with
    credentials that expired hours ago, and every archive read in sections 0,
    5, 8 and 9 would raise ``RequestExpired``/``ExpiredToken`` until the kernel
    is restarted. ``fresh_client`` builds a brand-new Session per call, which
    re-reads the credentials file and picks up rotated credentials on the very
    next call.

    It is reached through the module (``_aws.fresh_client``) rather than bound
    by name at import, matching ``tests/conftest.py::S3Archive``: this repo's
    AWS primitives are monkeypatched on the module, and a ``from ... import
    fresh_client`` here would hold a private reference the patch never reaches.
    """

    def __init__(self, uri: str, region: str | None = None) -> None:
        # parse_s3_uri is the results store's OWN parser. Sharing it is the
        # point: a second URI splitter here could disagree with the writers
        # that populate this prefix, and reader and writer would then orphan
        # history under a prefix neither of them looks in.
        self.bucket, self.prefix = parse_s3_uri(uri)
        self._client = _aws.fresh_client("s3", region)

    def _key(self, rel: str) -> str:
        rel = posixpath.normpath(rel)
        return f"{self.prefix}/{rel}" if self.prefix else rel

    def open(self, rel: str):
        """Return a streaming body for one object (read it, do not save it)."""
        try:
            return self._client.get_object(Bucket=self.bucket, Key=self._key(rel))["Body"]
        except self._client.exceptions.NoSuchKey as exc:
            raise FileNotFoundError(self._key(rel)) from exc

    def read(self, rel: str) -> bytes:
        return self.open(rel).read()

    def text(self, rel: str) -> str:
        return self.read(rel).decode("utf-8", errors="replace")

    def json(self, rel: str):
        return json.loads(self.text(rel))

    def size(self, rel: str) -> int:
        return int(self._client.head_object(Bucket=self.bucket, Key=self._key(rel))["ContentLength"])

    def sha256(self, rel: str) -> str:
        h = hashlib.sha256()
        for chunk in self.open(rel).iter_chunks(1 << 20):
            h.update(chunk)
        return h.hexdigest()


archive = S3Archive(ARCHIVE, ARCHIVE_REGION)
print("archive root:", ARCHIVE)

In [ ]:
"""Provenance: every archived object this notebook reads, with its sha256.

Executed live, on purpose. A number in sections 8 and 9 is only as good as
the bytes it came from, and this cell is the record of exactly which bytes
those were.
"""
#: archive-relative key -> which section consumes it.
ARCHIVE_INPUTS = {
    "notebooks/deduction/results/runs/flip_nemotron-3-nano-4b/flip_report.json":
        "section 8 -- score-level flip rate",
    "notebooks/deduction/results/runs/flip_nemotron-3-nano-4b/sample_manifest.json":
        "section 8 -- sample provenance (whitelist sha256, population size)",
    "notebooks/deduction/results/flip_free_bound_2026-08-18.json":
        "section 9 -- free flip bound",
    "notebooks/deduction/results/dojoinit_recovery_2026-08-18/report.json":
        "section 5 -- DojoInit recovery, sensitivity pool for error_bars",
}

print(f"{'sha256':>16} {'bytes':>10}  key")
print("-" * 100)
provenance = {}
for rel, use in ARCHIVE_INPUTS.items():
    digest, nbytes = archive.sha256(rel), archive.size(rel)
    provenance[rel] = {"sha256": digest, "bytes": nbytes, "used_by": use}
    print(f"{digest[:16]} {nbytes:>10}  {rel}")
    print(f"{'':>16} {'':>10}  -> {use}")
print("-" * 100)
print(f"{len(provenance)} archived object(s) resolved under {ARCHIVE}")

## Section 1 -- induction sizing (prospective)

**What this is.** The pre-registered replicate-sizing analysis for the
induction leg: per-family omnibus gates, the 210-contrast PRIMARY tier at
`ALPHA/210`, the 63-contrast SECONDARY tier under BH, and the recommended
`R`. It is **prospective only** -- it reads the *pilot* seed
(`PILOT_SEED = 0`) and deliberately refuses to read the collected block, so
sizing never becomes circular. The posterior counterpart is section 7.

**Inputs.** `notebooks/induction/results/<lane>_<arm>/rep_0.yaml`, which
`InductionExperiment.harness.sync_down()` writes from the S3 results store.

**Descends from.** The live module `notebooks/induction/analysis/power_analysis.py`
(imported above, *not* re-implemented).

In [ ]:
"""Induction replicate sizing. Needs the pilot replicate of every lane."""
if RUN_HEAVY:
    # ind_pa.RESULTS_DIR is anchored on the module's own __file__, so this
    # is notebooks/induction/results regardless of the notebook's cwd.
    print("results dir:", ind_pa.RESULTS_DIR)
    print("alpha primary:", ind_pa.ALPHA_PRIMARY, " alpha secondary:", ind_pa.ALPHA_SECONDARY)
    ind_pa.main()
else:
    print("skipped (RUN_HEAVY=False): needs notebooks/induction/results "
          "-- InductionExperiment.harness.sync_down() first")

## Section 2 -- induction paired re-analysis

**What this is.** The correction that produced the published induction
headline. The pre-registered test was an *unpaired* CMH, but both arms of a
contrast are drawn from the same replicate seeds, and a seed fixes the label
alphabet and answer vector shared by its 9 harmonics. So the notebook reports
three tests over the same 210 contrasts:

* `mcnemar_exact_p` -- exact conditional McNemar on the paired discordance;
* `signflip_exact_p` -- the **seed-level** exact sign-flip randomisation test,
  which treats the replicate (not the mark) as the exchangeable unit. This is
  the cluster-corrected primary; its resolution floor is `2 / 2**30` at R=30;
* `cmh_unpaired_p` -- the original statistic, kept so the paired-vs-unpaired
  difference isolates the *pairing* and nothing else;

plus `holm` (FWER over the 210-contrast family) and `design_effect`, the
observed/independence-assumed variance ratio of the per-seed arm difference:
the quantity CMH's denominator omits.

**Inputs.** `load_marks()` over every landed `rep_<seed>.yaml`.

**Descends from.** `notebooks/induction/analysis/paired_analysis.py`.

In [ ]:
"""Per-contrast paired statistics, built from paired_analysis' own primitives."""
if RUN_HEAVY:
    import numpy as np

    correct, valid = paired.load_marks()
    contrasts = ind_pa.build_primary_contrasts()
    assert len(contrasts) == ind_pa.N_PRIMARY

    rows = []
    for label, key_a, key_b in contrasts:
        a, b, seed_idx = paired.aligned(correct, valid, key_a, key_b, drop_invalid=False)
        disc_b = int((a & ~b).sum())
        disc_c = int((~a & b).sum())
        rows.append({
            "label": label,
            "n_items": int(a.size),
            "n_seeds": int(np.unique(seed_idx).size),
            "acc_a": float(a.mean()),
            "acc_b": float(b.mean()),
            "b": disc_b,
            "c": disc_c,
            "p_mcnemar": paired.mcnemar_exact_p(disc_b, disc_c),
            "p_signflip": paired.signflip_exact_p(paired.seed_diffs(a, b, seed_idx)),
            "p_cmh": paired.cmh_unpaired_p(a, b, seed_idx),
            "deff": paired.design_effect(a, b, seed_idx),
        })

    for stat in ("p_signflip", "p_mcnemar", "p_cmh"):
        rej = paired.holm(np.array([r[stat] for r in rows]), ind_pa.ALPHA)
        print(f"Holm rejections over {len(rows)} PRIMARY contrasts, {stat:>10}: "
              f"{int(rej.sum())}")

    deffs = np.array([r["deff"] for r in rows if r["deff"] is not None])
    print(f"design effect over {deffs.size} measurable contrasts: "
          f"median {np.median(deffs):.2f}, IQR "
          f"[{np.percentile(deffs, 25):.2f}, {np.percentile(deffs, 75):.2f}], "
          f"max {deffs.max():.2f}")
    print(f"(>1 means the unpaired CMH denominator is too small, i.e. "
          f"anticonservative)")
else:
    print("skipped (RUN_HEAVY=False): needs notebooks/induction/results")

In [ ]:
"""The canonical paired report: both invalid-handling passes, plus SECONDARY BH."""
if RUN_HEAVY:
    paired.main()
else:
    print("skipped (RUN_HEAVY=False): needs notebooks/induction/results")

## Section 3 -- induction significance report and the extens-vs-noise contrast

**What this is.** Two things, in the order the study needed them.

1. `significance_report.main()` -- the published induction headline. Holm
   at FWER 0.05 over the 210 PRIMARY contrasts (Hochberg runs alongside as a
   sensitivity check only), with the **collapse census**
   attached: a lane whose arm degenerated into repetition is reported as a
   first-class result with a mechanism annotation, never quarantined (user
   ruling, `no-quarantine-collapse-is-a-result`).
2. `extens_vs_noise.main()` -- the information-vs-length contrast, with the
   `mechanism()` classifier that separates a genuine information effect from
   a length/compliance artefact.

**Inputs.** `notebooks/induction/results`.

**Descends from.** The live modules `significance_report.py` and
`extens_vs_noise.py`.

> The three concluded audit probes that once ran here (`response_audit.py`,
> `verify_survivorship.py`, `check_currency.py`: the response-level audit
> behind the collapse annotations, ministral-3-14b's missing-not-at-random
> profile, and the content re-gate under earliest-wins) are archived, not
> tracked: `pr4_induction_audits_2026-08-30.zip` on the PR #4 release and
> `s3://smolbench-results-414266451290/archives/2026-08-30/`. See
> `notebooks/ARCHIVE.md`.


In [ ]:
"""Published induction significance report and the extens-vs-noise contrast."""
if RUN_HEAVY:
    significance.main()
    print("\n" + "=" * 100 + "\n")
    extens_vs_noise.main()
else:
    print("skipped (RUN_HEAVY=False): needs notebooks/induction/results")

## Section 4 -- deduction sizing (prospective)

**What this is.** The deduction leg's replicate-sizing analysis: for each of
the 21 within-family PRIMARY contrasts (and 63 cross-family SECONDARY ones),
a block bootstrap over **theorem blocks** gives the `n_theorems` power curve,
and a Beta-mixture projection gives the replicate sizing. Blocking is the
point: cells of one theorem are not independent draws, and `power_analysis`'s
own block bootstrap is this study's answer to that everywhere it matters.

**Inputs.** `<results-dir>/runs/scaling_<model>/verified_rows.jsonl` for all
21 lanes, or `--s3` to pull them from S3 into a temp dir -- by default the
re-collection's prefix (`LEAN_SPOOL_PREFIX`, or `deduction_postcutoff/runs`
if unset); reading the PUBLISHED pre-cutoff study means passing
`--spool-prefix deduction/runs` explicitly.

**Descends from.** `notebooks/deduction/analysis/power_analysis.py`. Note that
`power_analysis.py` is **not** what produced the published deduction
headline -- `error_bars.py` (section 5) is.

In [ ]:
"""Deduction replicate sizing over the 21 lanes."""
if RUN_HEAVY:
    print("primary alpha:", ded_pa.ALPHA_PRIMARY,
          " secondary alpha:", ded_pa.ALPHA_SECONDARY)
    # --s3 pulls verified_rows.jsonl for all 21 lanes into a temp dir it owns.
    # Swap for ["--results-dir", str(<dir>)] to analyse a local tree instead.
    rc = ded_pa.main(["--s3", "--sims", str(ded_pa.SIMS)])
    print("exit code:", rc)
else:
    print("skipped (RUN_HEAVY=False): needs the deduction run files "
          "(21 lanes x verified_rows.jsonl)")

## Section 5 -- deduction error bars (the published headline)

**What this is.** The statistic behind the study's published deduction
numbers. `build_pool` assembles the paired 21-way pool under one explicit
denominator rule (`count_as_failure=True`: a model-dependent no-survivor cell
scores 0 rather than dropping out); `block_matrix` reduces it to
`(n_theorems, n_models)` successes with a per-theorem cell count. Then:

* `bootstrap_stats` -- block bootstrap over theorem blocks with **BCa**
  intervals per lane (`B` up to 500k; a jackknife over blocks supplies the
  acceleration);
* `diff_ci` -- the paired difference interval, differenced *inside* each
  resample so the shared theorem draw cancels;
* `paired_mcnemar` -- cell-level exact McNemar for the same contrast;
* `block_signflip_p` -- the cluster-corrected primary: per-theorem
  differences with exchangeable signs, `(#{|perm| >= |obs|} + 1) / (B + 1)`;
* `holm` -- FWER over the 21 PRIMARY contrasts;
* `mode_report` -- the full table, plus the design effect against a naive
  binomial and the sensitivity pools under the other denominator rules.

**Inputs.** A **directory** of `<model>/verified_rows.jsonl` (`--rows-dir`)
and, for the sensitivity arm, a **directory** of
`<model>/recovered_rows.jsonl` (`--recovery-dir`).

> **Both are directory paths, and both now come from S3.**
> `build_pool` opens files off the filesystem; it has no streaming entry
> point, so both directories are fetched through
> `rows_source.resolve_rows_dir` into a fresh temporary directory --
> scratch, never the repo tree -- before `error_bars.main` ever opens a
> file. The first is this study's own spool prefix: the same 21 lanes'
> `verified_rows.jsonl` a plain `--s3` run reads. The second is the
> DojoInit recovery run, spooled under
> `<prefix>/dojoinit_recovery_2026-08-18/<lane>/recovered_rows.jsonl`;
> that same tree is ALSO mirrored into the **2026-08-25 archive**, but the
> cell below reads the SPOOL copy, not the archived one, because that is
> the location `scripts/results/audit_lean_pinning.py` constructs and can
> be checked against, and because rule 1 keeps archived data out of any
> local path -- the spool copy is fetched, never archived, so nothing
> here breaks that rule. The post-recovery sensitivity arm **is** computed
> below, as rows of `error_bars.main`'s own sensitivity table. What the
> ungated cell above does, separately, is stream the recovery's own
> `report.json` out of the archive and summarise it -- a pointer to the
> same run, not a substitute for computing the pool.

**Descends from.** `notebooks/deduction/analysis/error_bars.py`.

In [ ]:
"""Stream the DojoInit recovery report out of the archive (no download)."""
REC_REPORT = "notebooks/deduction/results/dojoinit_recovery_2026-08-18/report.json"
rec = archive.json(REC_REPORT)

print(f"{REC_REPORT}")
print(f"  sha256 {provenance[REC_REPORT]['sha256']}")


def _summarise(obj, indent=2, path=""):
    """Print a shallow, type-aware summary of a nested JSON report."""
    pad = " " * indent
    if isinstance(obj, dict):
        for key, val in obj.items():
            if isinstance(val, dict):
                print(f"{pad}{key}:")
                _summarise(val, indent + 2, f"{path}/{key}")
            elif isinstance(val, list):
                print(f"{pad}{key}: list[{len(val)}]")
            elif isinstance(val, str) and len(val) > 88:
                print(f"{pad}{key}: {val[:88]}...")
            else:
                print(f"{pad}{key}: {val}")


_summarise(rec)
print("\nThe post-recovery sensitivity pool computed in section 5's heavy")
print("cell below reads THIS SAME recovery run's rows -- fetched from its")
print("spool copy on S3 through rows_source, one directory over from the")
print("verified lanes -- rather than re-deriving anything from this report.json.")

In [ ]:
"""The published deduction error bars. Rows come from S3 through rows_source."""
if RUN_HEAVY:
    # a. The 21 lanes' verified rows, from this study's own spool prefix --
    # exactly what `error_bars.py --s3` reads.
    ROWS_DIR = rows_source.resolve_rows_dir(
        rows_dir=None, s3_prefix=rows_source.spool_prefix())

    # b. The DojoInit recovery run whose report.json the ungated cell above
    # summarises. That same tree is also mirrored into the 2026-08-25
    # archive, but the spool copy is read here instead of the archived one:
    # (i) it is the location in-tree code constructs and can be checked
    # against (scripts/results/audit_lean_pinning.py's fetch_recovery), and
    # (ii) rule 1 above keeps archived data out of any local path, and this
    # is a local path. These rows are FETCHED, not sha256-pinned, so they
    # are not in the provenance table in section 0.
    RECOVERY_RUN = "dojoinit_recovery_2026-08-18"
    RECOVERY_DIR = rows_source.resolve_rows_dir(
        rows_dir=None, s3_prefix=f"{rows_source.spool_prefix()}/{RECOVERY_RUN}",
        candidates=("recovered_rows.jsonl",), run_marker="")

    # c. A completeness gate before any pooling. error_bars.lane_outcomes
    # reads <recovery_dir>/<model>/recovered_rows.jsonl for EVERY model once
    # a recovery directory is given, so a lane missing from S3 must stop
    # this cell BY NAME rather than surface as a bare FileNotFoundError deep
    # inside the report -- or, worse, invite a "skip the missing lanes"
    # fallback: a 20-lane recovery pool compared against a 21-lane headline
    # is a wrong number that looks right.
    missing_recovery = sorted(
        m for m in ded_pa.MODELS
        if not (RECOVERY_DIR / m / "recovered_rows.jsonl").exists())
    if missing_recovery:
        raise SystemExit(
            f"DojoInit recovery rows missing for {missing_recovery} under "
            f"{RECOVERY_DIR} -- the post-recovery sensitivity arm needs all "
            f"{len(ded_pa.MODELS)} lanes, not a partial pool.")

    models, blocks, rungs, meta = error_bars.build_pool(
        ROWS_DIR, recovery_dir=None, count_as_failure=True)
    succ, size = error_bars.block_matrix(models, blocks)
    per_lane = dict(meta["own_rate"])
    print(f"pool: {succ.shape[0]} theorem blocks, {int(size.sum())} cells, "
          f"{len(models)} lanes")

    # The pieces, called directly, before the full report prints them.
    bs = error_bars.bootstrap_stats(succ, size, B=20_000,
                                    seed=error_bars.SIGNFLIP_SEED, alpha=0.05)
    contrasts = ded_pa.build_within_family_contrasts()
    p_signflip = error_bars.block_signflip_p(succ, models, contrasts)
    rej = error_bars.holm(p_signflip, error_bars.ALPHA)
    print(f"PRIMARY contrasts rejected under Holm (block sign-flip): "
          f"{int(rej.sum())} / {len(contrasts)}")
    # build_within_family_contrasts() yields (label, model_a, model_b) triples.
    for (label, a, b), p, r in list(zip(contrasts, p_signflip, rej))[:3]:
        d = error_bars.diff_ci(bs, models.index(a), models.index(b))
        nb, nc, p_mc = error_bars.paired_mcnemar(models, blocks, a, b)[:3]
        print(f"  {label}: diff {d['diff']:+.4f} "
              f"BCa [{d['lo']:+.4f}, {d['hi']:+.4f}]"
              f"{' (percentile fallback)' if d['fallback'] else ''}  "
              f"signflip p={p:.2e}  McNemar b/c={nb}/{nc} p={p_mc:.2e}  "
              f"Holm={'yes' if r else '.'}")

    # e. error_bars.main builds the three sensitivity pools itself (the two
    # denominator rules crossed with the recovery arm), so this cell stops
    # re-implementing the report's own logic; --rows-dir/--recovery-dir
    # point at what was already fetched above, so nothing downloads twice.
    rc = error_bars.main(["--rows-dir", str(ROWS_DIR),
                          "--recovery-dir", str(RECOVERY_DIR)])
    print("error_bars.main exit code:", rc)
else:
    print("skipped (RUN_HEAVY=False): a heavy run would fetch the 21 lanes' "
          "verified_rows.jsonl and the DojoInit recovery rows from S3")

## Section 6 -- deduction hint vs noise

**What this is.** The deduction leg's information-vs-length contrast.
`hint:3` and `noise:3` are byte-identical except for `hint:3`'s trailing
1-hop transitive premise-closure block, which `noise:3` replaces with
token-matched padding. So this tests *supplementary* background on top of an
already-complete direct-premise context -- **not** the same manipulation as
the induction `extens`-vs-`noise` contrast, and the report says so. One cell
per theorem per model, so exact McNemar applies with no cluster correction,
and Holm runs over the 21 models. The report also states the minimum
detectable effect, so a null is not read as an absence of measurement.

**Inputs.** The same `--rows-dir` directory of `<model>/verified_rows.jsonl`
as section 5.

**Descends from.** `notebooks/deduction/analysis/hint_vs_noise.py`.

In [ ]:
"""hint:3 vs noise:3, per model, paired within theorem."""
if RUN_HEAVY:
    # Section 5 already fetched this exact <model>/verified_rows.jsonl
    # tree; a second --s3 here would pull all 21 lanes twice for one
    # report, so this reuses ROWS_DIR instead.
    rc = hint_vs_noise.main(["--rows-dir", str(ROWS_DIR)])
    print("exit code:", rc)
else:
    print("skipped (RUN_HEAVY=False): needs the same rows directory as "
          "section 5 (fetched from S3)")

## Section 7 -- posterior power

**What this is.** The *posterior* counterpart to sections 1 and 4: at the R
already collected, which planned contrasts are settled and which would
actually benefit from more data.

It is deliberately **not** "observed power". Observed power is a monotone
function of the p-value, so it carries no information beyond it: a
non-significant result always yields low observed power, and reporting it
dresses "p was large" up as independent evidence. Instead every contrast
lands in one of three states:

| state | rule | meaning |
|---|---|---|
| `DECIDED` | test rejects at the corrected alpha | settled; more replicates cannot unsettle it |
| `EQUIVALENT` | not significant **and** the CI lies entirely inside +-MEI | a demonstrated near-tie; also settled |
| `UNDECIDED` | not significant **and** the CI still spans MEI | the only state where more data helps |

**MEI is pre-specified**, not observed: a claim that a difference is too
small to care about must say in advance how big "care about" is. The
equivalence interval is a `1 - 2*alpha` interval (the TOST convention: such
an interval inside +-MEI is exactly two one-sided tests both rejecting at
alpha).

`MIN_R_FOR_EQUIVALENCE = 5` guards the one way this can lie: a bootstrap over
2 agreeing replicates has near-zero width, fits inside any MEI, and
manufactures an equivalence claim out of almost no data. Below the threshold
the honest state is `UNDECIDED`.

**Where each statistic comes from.** The classifier, the MEI framing and the
`MIN_R_FOR_EQUIVALENCE` guard are implemented below. Everything else comes
from the live modules: `paired_analysis.cmh_unpaired_p` for the p-value,
`error_bars.bootstrap_stats` + `error_bars.diff_ci` for the paired BCa
interval, `power_analysis.replicates_needed` for sizing, and
`paired_analysis.load_marks` for loading (it also returns the validity mask
that keeps invalid marks distinguishable from wrong ones).

**The family changes with the roster.** Parameterised by the *current*
21-lane roster from `notebooks/induction/run_study.py`, the all-pairs
construction gives `4*C(21,2) + 21*C(4,2) = 966` tests and a far stricter
alpha. That family is also **not** the study's pre-registered one: the study
registers 210 PRIMARY within-family ladder contrasts at `ALPHA/210`
(`power_analysis.N_PRIMARY`). Both are computed below, and the report states
which alpha it used.

**Inputs.** `notebooks/induction/results` (gated). The classifier itself is
pure and is exercised on synthetic counts, ungated.

In [ ]:
"""Posterior-power port: the classifier, the MEI framing, and the R guard."""
import math
from itertools import combinations

import numpy as np

#: Minimum replicates per side before an EQUIVALENT verdict is allowed
#: Equivalence is a positive claim: it needs
#: enough replicates to have been able to refute itself.
MIN_R_FOR_EQUIVALENCE = 5

#: Default pre-specified minimum effect of interest, absolute accuracy.
#: Pre-specify it; do not tune it to the data.
DEFAULT_MEI = 0.05

#: Resamples wanted in EACH tail before an interval endpoint is an ESTIMATE of
#: a quantile rather than the extreme order statistic. Below this the endpoint
#: simply IS the most extreme draw, which sits inward of the true quantile: at
#: the fixed B = 4,000 this cell used to carry, the posterior family's alpha
#: put an expected 0.2 resamples in the tail being read. An inward endpoint
#: narrows the interval, a narrower interval is likelier to sit inside +-MEI,
#: and so the bias runs toward EQUIVALENT -- the one verdict that CLOSES a
#: contrast.
BOOT_TAIL_TARGET = 50

#: Ceiling on the derived count, imposed by cost and nothing else. One
#: `paired_diff_ci` call measured 4.2 ms at B=4,000, 31 ms at B=50,000 and
#: 121 ms at this cap; section 7's real-data pass runs 966 contrasts, so the
#: cap takes that pass from about 4 s to about 2 minutes. Reaching the cap
#: means the `BOOT_TAIL_TARGET` criterion is NOT met -- the cap is a cost
#: ceiling, not a resolution fix, which is why `boot_resamples` says so aloud.
BOOT_RESAMPLE_CAP = 200_000

#: Two-sided alphas already warned about. Section 7 derives a count once per
#: contrast, so without this ledger the capped-alpha warning would print 966
#: times and be scrolled past instead of read.
_BOOT_CAP_WARNED: set[float] = set()


def posterior_family(models, infos) -> int:
    """Count the all-pairs contrast family: model-within-info plus info-within-model.

    This includes the chance-floor ``zero`` contrasts on purpose. They are
    trivially significant, but dropping them from the correction after
    seeing the data is exactly the multiplicity abuse the correction
    exists to prevent.
    """
    return len(infos) * len(list(combinations(models, 2))) + \
        len(models) * len(list(combinations(infos, 2)))


def build_posterior_contrasts(models, infos):
    """Build every (label, key_a, key_b) in the all-pairs posterior family."""
    out = []
    for info in infos:
        for m_a, m_b in combinations(models, 2):
            out.append((f"[{info}] {m_a} vs {m_b}", (m_a, info), (m_b, info)))
    for model in models:
        for i_a, i_b in combinations(infos, 2):
            out.append((f"[{model}] {i_a} vs {i_b}", (model, i_a), (model, i_b)))
    return out


def classify(p, ci_lo, ci_hi, mei, r_min, alpha,
             min_r=MIN_R_FOR_EQUIVALENCE) -> str:
    """Sort one contrast into DECIDED / EQUIVALENT / UNDECIDED.

    Parameters
    ----------
    p : float
        The contrast's test p-value (here: `paired_analysis.cmh_unpaired_p`).
    ci_lo, ci_hi : float
        Bounds of the ``1 - 2*alpha`` interval for the accuracy difference.
    mei : float
        Pre-specified minimum effect of interest, absolute accuracy.
    r_min : int
        Replicates on the SMALLER side of the contrast.
    alpha : float
        Multiplicity-corrected significance threshold.
    min_r : int, default MIN_R_FOR_EQUIVALENCE
        Replicates required before EQUIVALENT is allowed.

    Returns
    -------
    str
        ``"DECIDED"``, ``"EQUIVALENT"`` or ``"UNDECIDED"``. A contrast that
        would be EQUIVALENT on interval width alone, but has fewer than
        `min_r` replicates, is UNDECIDED: that is the honest state, not a
        near-tie.
    """
    if p < alpha:
        return "DECIDED"
    if ci_lo > -mei and ci_hi < mei and r_min >= min_r:
        return "EQUIVALENT"
    return "UNDECIDED"


def boot_resamples(alpha, target=BOOT_TAIL_TARGET, cap=BOOT_RESAMPLE_CAP) -> int:
    """Derive the bootstrap resample count from the alpha being read.

    Parameters
    ----------
    alpha : float
        The TWO-SIDED level, exactly as `error_bars.bootstrap_stats` takes it.
        That function cuts each tail at ``alpha / 2``, so the criterion is
        applied per TAIL -- using `alpha` itself would ask for half the
        resolution actually needed.
    target : int, default BOOT_TAIL_TARGET
        Resamples wanted in each tail.
    cap : int, default BOOT_RESAMPLE_CAP
        Ceiling on the returned count.

    Returns
    -------
    int
        ``ceil(target / (alpha / 2))``, or `cap` when that exceeds it. A capped
        return is a FLOOR on what the interval resolves, never a fix: the
        endpoints still do not resolve `alpha`, and the caller is told so.

    Notes
    -----
    Prints when it caps, rather than calling ``warnings.warn``. This notebook
    communicates by printing, and a warning suppressed by the kernel's filter
    configuration would turn the one honest signal here into a silent
    fallback. It prints once per distinct `alpha` (`_BOOT_CAP_WARNED`).

    Examples
    --------
    >>> boot_resamples(0.01)
    10000
    """
    tail = alpha / 2
    wanted = math.ceil(target / tail)
    if wanted <= cap:
        return wanted
    if alpha not in _BOOT_CAP_WARNED:
        _BOOT_CAP_WARNED.add(alpha)
        print(
            f"WARNING: alpha={alpha:.6g} wants {wanted} resamples for {target} "
            f"draws in each tail; capping at {cap}. The interval endpoints are "
            f"NOT resolved to this alpha -- at the cap only {cap * tail:.3g} "
            f"resamples land in the tail an endpoint is read from. Raising the "
            f"cap would not fix it: see the resample sweep below, where no B on "
            f"error_bars.B_GRID reaches error_bars.DRIFT_TOL at this alpha, and "
            f"the R = 30 block-count note beside it for why."
        )
    return cap


def paired_diff_ci(a, b, seed_idx, alpha, n_boot=None, seed=0) -> dict:
    """Bootstrap the accuracy difference ``a - b``, resampling REPLICATES.

    The replicate is the independent unit; harmonics inside one are strata
    of differing difficulty, not exchangeable draws, so they are never
    resampled. This delegates to `error_bars.bootstrap_stats`, treating
    each replicate as a "block" and the two arms as two "models", so the
    difference is taken INSIDE each resample and the shared replicate draw
    cancels (which the archived script's independent two-arm resampling
    did not do).

    Parameters
    ----------
    n_boot : int, optional
        Resamples. ``None`` (the default) derives the count from `alpha` via
        `boot_resamples`, so the interval's Monte-Carlo resolution follows the
        level being read instead of a fixed number that silently under-resolves
        a small alpha. A caller may still pin it, and `resample_sweep` does --
        that is how the drift across `error_bars.B_GRID` gets measured.

    Returns
    -------
    dict
        `error_bars.diff_ci`'s dict: ``diff`` (= mean(a) - mean(b)),
        ``lo``, ``hi``, ``se``, ``fallback``.
    """
    if n_boot is None:
        n_boot = boot_resamples(alpha)
    seeds = np.unique(seed_idx)
    succ = np.array([[a[seed_idx == s].sum(), b[seed_idx == s].sum()] for s in seeds],
                    dtype=float)
    size = np.array([(seed_idx == s).sum() for s in seeds], dtype=float)
    bs = error_bars.bootstrap_stats(succ, size, B=n_boot, seed=seed, alpha=alpha)
    # diff_ci(bs, ja, jb) is rate(jb) - rate(ja); column 0 is arm a, so
    # (ja=1, jb=0) gives a - b.
    return error_bars.diff_ci(bs, 1, 0)


# --- the current roster, and the two alphas it implies ----------------------
# run_study.MODELS maps SPEC KEY ("qwen3.5-27b") -> LOCAL RESULTS TAG
# ("qwen35_27b"). The analysis modules are keyed by the TAG -- load_marks
# reads RESULTS_DIR / f"{tag}_{info}" -- so the roster is taken through the
# mapping. The assert below pins
# the two name spaces together: a roster edit on one side that does not land
# on the other must fail loudly here, not silently skip every contrast.
ROSTER_SPECS = tuple(run_study.MODELS)                  # declaration order
ROSTER_MODELS = tuple(run_study.MODELS[k] for k in ROSTER_SPECS)   # local tags
ROSTER_INFOS = tuple(run_study.INFO_TYPES)       # intens / extens / noise_intens / zero
assert set(ROSTER_MODELS) == set(ind_pa.MODELS), (
    "run_study's local result tags and power_analysis.MODELS disagree: "
    f"{sorted(set(ROSTER_MODELS) ^ set(ind_pa.MODELS))}")
N_TESTS = posterior_family(ROSTER_MODELS, ROSTER_INFOS)
ALPHA_POSTERIOR = power_common.ALPHA / N_TESTS

print(f"roster: {len(ROSTER_MODELS)} lanes x {len(ROSTER_INFOS)} info arms "
      f"(notebooks/induction/run_study.py)")
print(f"  spec key -> results tag, e.g. {ROSTER_SPECS[0]} -> {ROSTER_MODELS[0]}"
      f"  (matches power_analysis.MODELS)")
print(f"all-pairs posterior family : N_TESTS = {N_TESTS}, "
      f"alpha = {power_common.ALPHA}/{N_TESTS} = {ALPHA_POSTERIOR:.3e}")
print(f"pre-registered PRIMARY family: N = {ind_pa.N_PRIMARY}, "
      f"alpha = {ind_pa.ALPHA_PRIMARY:.3e} -- a DIFFERENT family "
      f"(within-family ladder contrasts only)")
print(f"MIN_R_FOR_EQUIVALENCE = {MIN_R_FOR_EQUIVALENCE}, default MEI = {DEFAULT_MEI}")

In [ ]:
"""Self-test: exercise the ported classifier on synthetic counts. Ungated."""
import numpy as np

_MEI, _ALPHA = 0.05, 0.05

#: SD of the arm-specific per-replicate logit offset `synth` applies in its
#: clustered case. Calibrated against the LIVE metric rather than guessed: at
#: R=40 it puts `paired.design_effect` -- the same statistic section 2 reports
#: on the real induction data -- at a median of about 3.0, the middle of the
#: range seen there. Median deff over 200 draws by cluster_sd: 0.9 -> 2.19,
#: 1.3 -> 2.96, 1.4 -> 3.04, 1.5 -> 3.25.
CLUSTER_SD = 1.4

# 1. The pure decision table, including the guard branch that the real data
#    can never reach: every lane collected R = 30, so r_min < 5 never occurs
#    outside this test.
CASES = [
    # (p, lo, hi, r_min, expected, what it pins)
    (1e-9, +0.20, +0.40, 30, "DECIDED", "rejects: settled whatever the CI"),
    (1e-9, -0.01, +0.01, 3, "DECIDED", "rejection outranks the R guard"),
    (0.42, -0.02, +0.03, 30, "EQUIVALENT", "CI inside +-MEI, enough replicates"),
    (0.42, -0.02, +0.03, 3, "UNDECIDED", "same CI, too few replicates: the guard"),
    (0.42, -0.02, +0.03, 5, "EQUIVALENT", "exactly at the guard threshold"),
    (0.42, -0.09, +0.02, 30, "UNDECIDED", "CI still spans -MEI"),
    (0.42, -0.02, +0.09, 30, "UNDECIDED", "CI still spans +MEI"),
    (0.42, -0.05, +0.05, 30, "UNDECIDED", "CI touching +-MEI is NOT equivalence"),
]
for p, lo, hi, r_min, expected, why in CASES:
    got = classify(p, lo, hi, _MEI, r_min, _ALPHA)
    assert got == expected, f"classify({p}, {lo}, {hi}, r_min={r_min}) = {got} != {expected}"
    print(f"  ok  {got:<10} r_min={r_min:<3} CI [{lo:+.2f},{hi:+.2f}] p={p:<8.3g} {why}")

# 2. End to end on synthetic marks, through the LIVE modules the port reuses:
#    paired_analysis.cmh_unpaired_p for p, error_bars.bootstrap_stats/diff_ci
#    for the interval.
rng = np.random.default_rng(0)
N_HARM = ind_pa.N_HARMONICS


def synth(rate, n_seeds, gen, cluster_sd=0.0):
    """Draw an (n_seeds * N_HARM) flat mark vector plus its replicate index.

    Parameters
    ----------
    rate : float
        Marginal success rate. With `cluster_sd` > 0 it must lie strictly
        inside ``(0, 1)``, since the offset is applied on the logit scale.
    n_seeds : int
        Replicates to draw. Each contributes ``N_HARM`` marks.
    gen : numpy.random.Generator
        Drawn from in place; the caller owns the stream.
    cluster_sd : float, default 0.0
        SD of an arm-specific per-replicate LOGIT offset, shared by all
        ``N_HARM`` harmonics of that replicate. The two arms of a contrast
        call this separately and so draw their offsets INDEPENDENTLY, which is
        what makes the effect arm-specific rather than a common difficulty
        term that would cancel in the difference.

    Returns
    -------
    tuple of (ndarray, ndarray)
        The flat 0/1 mark vector and its replicate index, in the shape
        `paired_diff_ci` and `paired.cmh_unpaired_p` take.

    Raises
    ------
    ValueError
        `cluster_sd` negative, or `rate` outside ``(0, 1)`` when clustering is
        requested (``logit(0)`` and ``logit(1)`` are infinite).

    Notes
    -----
    Clustering is the shape the study's data actually has: harmonics inside a
    replicate are strata of differing difficulty, not exchangeable draws --
    which is `paired_diff_ci`'s own stated reason for resampling REPLICATES
    and never harmonics. Drawing them i.i.d. tests the classifier on data the
    study does not collect.

    ``cluster_sd=0`` is the exact i.i.d. behaviour this function had before
    clustering existed, and it takes NO extra draw from `gen`: the branch is
    on the parameter, not on a zero-scale normal, because ``normal(0, 0)``
    still advances the stream. The self-test threads ONE generator through its
    cases in order, so an extra draw here would shift every case after it.
    """
    if cluster_sd < 0:
        raise ValueError(f"synth: cluster_sd must be >= 0, got {cluster_sd}")
    if cluster_sd == 0:
        # No call to `gen` beyond this one -- see Notes.
        marks = (gen.random((n_seeds, N_HARM)) < rate)
    else:
        if not 0.0 < rate < 1.0:
            raise ValueError(
                "synth: clustering applies the offset on the logit scale, so "
                f"rate must be strictly inside (0, 1), got {rate}"
            )
        offset = gen.normal(0.0, cluster_sd, size=n_seeds)
        p_rep = 1.0 / (1.0 + np.exp(-(np.log(rate / (1.0 - rate)) + offset)))
        marks = (gen.random((n_seeds, N_HARM)) < p_rep[:, None])
    seed_idx = np.repeat(np.arange(n_seeds), N_HARM)
    return marks.reshape(-1), seed_idx


def posterior_state(a, b, seed_idx, mei, alpha, n_seeds):
    """Run one synthetic contrast through the whole ported pipeline."""
    p = paired.cmh_unpaired_p(a, b, seed_idx)
    ci = paired_diff_ci(a, b, seed_idx, alpha=2 * alpha, seed=0)
    return p, ci, classify(p, ci["lo"], ci["hi"], mei, n_seeds, alpha)


# 2a. A large true effect -> DECIDED.
a, si = synth(0.95, 12, rng)
b, _ = synth(0.15, 12, rng)
p, ci, state = posterior_state(a, b, si, _MEI, _ALPHA, 12)
print(f"\n  synthetic 0.95 vs 0.15, R=12: diff {ci['diff']:+.3f} "
      f"CI [{ci['lo']:+.3f}, {ci['hi']:+.3f}] p={p:.2e} -> {state}")
assert state == "DECIDED", state

# 2b. A true null at a generous MEI with enough replicates -> EQUIVALENT.
a, si = synth(0.50, 40, rng)
b, _ = synth(0.50, 40, rng)
p, ci, state = posterior_state(a, b, si, 0.15, _ALPHA, 40)
print(f"  synthetic 0.50 vs 0.50, R=40, MEI=0.15: diff {ci['diff']:+.3f} "
      f"CI [{ci['lo']:+.3f}, {ci['hi']:+.3f}] p={p:.3f} -> {state}")
assert state == "EQUIVALENT", state

# 2c. The SAME null on 3 replicates -> UNDECIDED, via the guard alone.
a3, si3 = synth(0.50, 3, rng)
b3, _ = synth(0.50, 3, rng)
p3, ci3, state3 = posterior_state(a3, b3, si3, 0.30, _ALPHA, 3)
print(f"  synthetic 0.50 vs 0.50, R=3,  MEI=0.30: diff {ci3['diff']:+.3f} "
      f"CI [{ci3['lo']:+.3f}, {ci3['hi']:+.3f}] p={p3:.3f} -> {state3} "
      f"(MIN_R_FOR_EQUIVALENCE={MIN_R_FOR_EQUIVALENCE})")
assert state3 == "UNDECIDED", state3

# 2d. The SAME true null as 2b, but with an arm-specific per-replicate effect --
#     the shape paired_diff_ci's docstring says the real data has. This case is
#     REPORTED and deliberately NOT asserted: under clustering the verdict is
#     not stable, and asserting EQUIVALENT here was measured failing 38 times in
#     60 at deff 3.19. One draw is an illustration, not evidence; the
#     verdict-distribution cell at the end of this section is the evidence.
#     Its own generator, so this block cannot perturb any stream above it.
cgen = np.random.default_rng(20260904)
ac, sic = synth(0.50, 40, cgen, cluster_sd=CLUSTER_SD)
bc, _ = synth(0.50, 40, cgen, cluster_sd=CLUSTER_SD)
pc, cic, statec = posterior_state(ac, bc, sic, 0.15, _ALPHA, 40)
deffc = paired.design_effect(ac, bc, sic)
# design_effect returns None for "no measurable ratio"; print that, never a
# stand-in value that would read as a measurement.
deff_txt = f"{deffc:.2f}" if deffc is not None else "None (no measurable ratio)"
print(f"  synthetic 0.50 vs 0.50, R=40, MEI=0.15, cluster_sd={CLUSTER_SD}: "
      f"diff {cic['diff']:+.3f} "
      f"CI [{cic['lo']:+.3f}, {cic['hi']:+.3f}] p={pc:.3f} deff={deff_txt} "
      f"-> {statec}   [REPORTED, not asserted]")

print("\nposterior-power port: self-test PASSED "
      f"({len(CASES)} decision cases + 3 asserted i.i.d. end-to-end contrasts).")
print("The clustered contrast above is REPORTED, not asserted -- PASSED does "
      "NOT cover it.\nSee the verdict-distribution cell at the end of this "
      "section for what clustering costs.")

### How many bootstrap resamples -- and what B cannot buy

**What this is.** `paired_diff_ci` used to draw a fixed 4,000 resamples
whatever alpha it was asked about. At the posterior family's
`alpha = 0.05/966 = 5.18e-05` per tail that put an expected **0.2** resamples
in the tail an endpoint is read from, so the endpoint was not an estimated
quantile at all: it was the single most extreme draw, which sits inward of the
true quantile. Inward endpoints narrow the interval, a narrower interval is
likelier to fall inside `+-MEI`, and so the bias runs toward EQUIVALENT -- the
one verdict that closes a contrast. `boot_resamples` now derives B from the
alpha in use, targeting `BOOT_TAIL_TARGET` draws in each tail.

**What B buys.** Monte-Carlo error, and nothing else -- the difference between
this resample draw and another one. `resample_sweep` below measures exactly
that across `error_bars.B_GRID`, against `error_bars.DRIFT_TOL` (0.0005
accuracy points: rates are printed to 3 decimals, so drift under half a
thousandth cannot change a reported figure). `error_bars.py` chooses its own B
by this same criterion on this same grid, which is why both are referenced
through the module instead of re-declared here.

**The measured answer: no B on the grid is enough.** At this alpha every
`B_GRID` entry drifts by an order of magnitude more than `DRIFT_TOL`, and the
largest of them, 500,000, still puts only 25.9 draws in a tail that wants 50.
`BOOT_RESAMPLE_CAP` is therefore a cost ceiling and not a resolution fix: at
121 ms per contrast the cap already costs section 7's real-data pass about two
minutes over 966 contrasts, and buying the full 966,000 would cost roughly ten
and still not reach `DRIFT_TOL`. Read the EQUIVALENT verdicts at this alpha as
approximate.

**Why more resamples cannot rescue it: R = 30 blocks.** The binding limit is
the data, not the arithmetic.

1. The bootstrap resamples REPLICATES, so every resample statistic is a convex
   combination of the same R = 30 per-replicate differences. An interval
   endpoint can therefore never leave the `[min, max]` of those 30 numbers,
   whatever B is -- the cell below prints that range for the contrast it
   sweeps.
2. BCa's bias-correction and acceleration terms are estimated from a 30-point
   jackknife, so the correction applied to the endpoints is itself as coarse
   as the sample it comes from.
3. A `1 - 2*alpha` interval at `alpha = 5.18e-05` asks for the 0.0052nd
   percentile of a distribution built from 30 observations. That is
   extrapolation past the smallest thing this many blocks can resolve, not
   estimation, and no amount of B changes it.

In [ ]:
"""Resample-count sweep: how much of the interval is Monte-Carlo noise. Ungated."""
import numpy as np


def resample_sweep(a, b, seed_idx, alpha, grid=None) -> list[dict]:
    """Measure interval-endpoint drift across a grid of resample counts.

    Mirrors `error_bars.mode_sweep`: each B runs on its OWN RNG stream, so the
    drift between two rows measures Monte-Carlo error rather than one seed's
    luck. Sharing a stream would correlate consecutive rows and make the drift
    read smaller than it is.

    Parameters
    ----------
    a, b : ndarray
        Flat 0/1 mark vectors for the two arms, as `paired_diff_ci` takes them.
    seed_idx : ndarray
        Replicate index per mark; the replicate is the resampled block.
    alpha : float
        Two-sided level, passed straight through to `paired_diff_ci`.
    grid : sequence of int, optional
        Resample counts to try, ascending. ``None`` uses `error_bars.B_GRID` --
        the grid `error_bars.py` chooses its own B on, referenced rather than
        copied so the two cannot drift apart.

    Returns
    -------
    list of dict
        One row per grid entry, in grid order, with keys ``B``, ``lo``, ``hi``,
        ``drift`` and ``expected_tail``. ``drift`` is the largest absolute
        endpoint move against the PREVIOUS (smaller) B; it is ``None`` on the
        first row, which has nothing to compare against -- ``None`` rather than
        0.0, so "not compared" cannot be misread as "did not move".
        ``expected_tail`` is ``B * alpha / 2``, the expected number of
        resamples landing in the tail an endpoint is read from.

    Notes
    -----
    Costs one `paired_diff_ci` per grid entry; over the full `error_bars.B_GRID`
    (876,000 resamples) that is about half a second on R = 30 blocks.
    """
    grid = error_bars.B_GRID if grid is None else grid
    rows: list[dict] = []
    prev = None
    for k, n_boot in enumerate(grid):
        # seed = 1000 + k, the offset error_bars.mode_sweep uses: an independent
        # stream per B is what makes the drift a Monte-Carlo measurement.
        ci = paired_diff_ci(a, b, seed_idx, alpha=alpha, n_boot=n_boot, seed=1000 + k)
        drift = (None if prev is None
                 else max(abs(ci["lo"] - prev[0]), abs(ci["hi"] - prev[1])))
        rows.append({"B": n_boot, "lo": ci["lo"], "hi": ci["hi"], "drift": drift,
                     "expected_tail": n_boot * alpha / 2})
        prev = (ci["lo"], ci["hi"])
    return rows


# One R = 30 TRUE NULL contrast: the study's own replicate count, and the case
# where an inward-biased endpoint does the most damage -- a true null is where
# EQUIVALENT ought to be reachable honestly. Fixed seed so the table below is
# reproducible; a private generator name so nothing downstream shares its stream.
_sweep_gen = np.random.default_rng(20260904)
_sw_a, _sw_idx = synth(0.5, 30, _sweep_gen)
_sw_b, _ = synth(0.5, 30, _sweep_gen)
_sw_alpha = 2 * ALPHA_POSTERIOR

print(f"Resample-count sweep -- R = {np.unique(_sw_idx).size} replicate blocks, "
      f"alpha = {_sw_alpha:.3e} two-sided ({_sw_alpha / 2:.3e} per tail)")
print("Each B runs on an INDEPENDENT RNG stream; drift = max |endpoint change| "
      "vs the\nnext SMALLER B.\n")
print(f"{'B':>8s} {'lo':>10s} {'hi':>10s} {'drift (pts)':>13s} "
      f"{'vs DRIFT_TOL':>13s} {'exp. draws/tail':>17s}")
print("-" * 76)
_sw_rows = resample_sweep(_sw_a, _sw_b, _sw_idx, alpha=_sw_alpha)
for _sw_row in _sw_rows:
    _sw_drift = "(baseline)" if _sw_row["drift"] is None else f"{_sw_row['drift']:.5f}"
    _sw_flag = ("" if _sw_row["drift"] is None
                else ("OK" if _sw_row["drift"] <= error_bars.DRIFT_TOL else "OVER"))
    print(f"{_sw_row['B']:8d} {_sw_row['lo']:10.4f} {_sw_row['hi']:10.4f} "
          f"{_sw_drift:>13s} {_sw_flag:>13s} {_sw_row['expected_tail']:17.2f}")

# The verdict, stated rather than left to the reader to infer from the column.
_sw_met = [r["B"] for r in _sw_rows
           if r["drift"] is not None and r["drift"] <= error_bars.DRIFT_TOL]
print(f"\nTolerance: {error_bars.DRIFT_TOL} pts. " + (
    f"First met at B={min(_sw_met)}." if _sw_met else
    f"NO B on error_bars.B_GRID meets it at this alpha.\nThe largest, "
    f"{_sw_rows[-1]['B']}, still puts only {_sw_rows[-1]['expected_tail']:.1f} "
    f"resamples in a tail that wants {BOOT_TAIL_TARGET}, and its endpoints are "
    f"still moving\n{_sw_rows[-1]['drift'] / error_bars.DRIFT_TOL:.0f}x the "
    f"tolerance -- what is short here is the tail count, not the grid."))

# The block-count limit, measured rather than asserted: every resample statistic
# is a convex combination of these R = 30 per-replicate differences, so no B can
# move an endpoint outside their range.
_sw_per_rep = np.array([_sw_a[_sw_idx == s].mean() - _sw_b[_sw_idx == s].mean()
                        for s in np.unique(_sw_idx)])
print(f"Per-replicate differences over the {_sw_per_rep.size} blocks span "
      f"[{_sw_per_rep.min():+.4f}, {_sw_per_rep.max():+.4f}]. Every resample "
      f"statistic is a convex\ncombination of them, so no B moves an endpoint "
      f"outside that range -- which is the\nR = 30 limit the note above "
      f"describes, and the reason the cap is a cost ceiling only.")

del _sweep_gen, _sw_a, _sw_b, _sw_idx, _sw_alpha, _sw_rows, _sw_row, _sw_drift
del _sw_flag, _sw_met, _sw_per_rep

In [ ]:
"""Posterior power over the collected block. Needs the induction results tree."""
if RUN_HEAVY:
    import numpy as np

    MEI = DEFAULT_MEI
    ALPHA_USED = ALPHA_POSTERIOR      # all-pairs family; see this section's note

    correct, valid = paired.load_marks()
    # Gate on CONTENT: a roster/name-space mismatch must stop the section, not
    # quietly print a table of zeros.
    missing = [key for _, key_a, key_b in
               build_posterior_contrasts(ROSTER_MODELS, ROSTER_INFOS)
               for key in (key_a, key_b) if key not in correct]
    if missing:
        raise SystemExit(f"{len(set(missing))} roster condition(s) absent from "
                         f"load_marks(), e.g. {sorted(set(missing))[:3]}")
    invalid_counts = {
        key: int(sum((~v).sum() for v in valid[key].values())) for key in valid
    }

    print(f"MEI {MEI:.3f} absolute accuracy   alpha {power_common.ALPHA}/{N_TESTS} "
          f"= {ALPHA_USED:.2e}")
    print(f"\n{'contrast':>46} {'diff':>8} {'1-2a CI':>20} {'p (CMH)':>10}  state")
    print("-" * 104)

    tally = {"DECIDED": 0, "EQUIVALENT": 0, "UNDECIDED": 0, "SKIPPED": 0}
    undecided = []
    for label, key_a, key_b in build_posterior_contrasts(ROSTER_MODELS, ROSTER_INFOS):
        if key_a not in correct or key_b not in correct:
            tally["SKIPPED"] += 1
            continue
        a, b, seed_idx = paired.aligned(correct, valid, key_a, key_b, drop_invalid=False)
        n_seeds = int(np.unique(seed_idx).size)
        p = paired.cmh_unpaired_p(a, b, seed_idx)
        ci = paired_diff_ci(a, b, seed_idx, alpha=2 * ALPHA_USED)
        state = classify(p, ci["lo"], ci["hi"], MEI, n_seeds, ALPHA_USED)
        tally[state] += 1
        if state == "UNDECIDED":
            undecided.append((label, key_a, key_b, n_seeds))
        print(f"{label:>46} {ci['diff']:>+8.4f} "
              f"[{ci['lo']:>+8.4f},{ci['hi']:>+8.4f}] {p:>10.2e}  {state}")

    print("-" * 104)
    print(f"DECIDED {tally['DECIDED']}   EQUIVALENT {tally['EQUIVALENT']}   "
          f"UNDECIDED {tally['UNDECIDED']}   SKIPPED {tally['SKIPPED']}")
    print(f"invalid marks (score: null), counted as failures above: "
          f"{sum(invalid_counts.values())} over {len(invalid_counts)} conditions")

    if not undecided:
        print(f"\nNOTHING FURTHER NEEDED at MEI={MEI:.3f}: every contrast is "
              "resolved or demonstrated equivalent.")
    else:
        # Sizing runs at the MEI, not at the observed effect. Sizing from an
        # observed effect is biased: the same noise that made the effect look
        # large is what selected it into this list.
        # COST NOTE: at this alpha a contrast that never reaches 80% power
        # scans all MAX_REPLICATES values x N_SIMS simulations, twice per
        # contrast. Expect minutes per undecided contrast.
        rng = np.random.default_rng(power_common.SEED)
        print(f"\n{len(undecided)} undecided -- replicates needed "
              f"(R at MEI is the honest target; R at observed is context only):")
        print(f"\n{'contrast':>46} {'R now':>6} {'R for MEI':>10} {'R at observed':>14}")
        for label, key_a, key_b, n_seeds in undecided:
            base = np.array([np.mean([correct[key_a][s][k] for s in correct[key_a]])
                             for k in range(ind_pa.N_HARMONICS)])
            other = np.array([np.mean([correct[key_b][s][k] for s in correct[key_b]])
                              for k in range(ind_pa.N_HARMONICS)])
            shifted = np.clip(base - MEI, 0.0, 1.0)
            # replicates_needed returns (needed, curve) over POWER_TARGETS.
            need_mei, _ = ind_pa.replicates_needed(base, shifted, rng, alpha=ALPHA_USED)
            need_obs, _ = ind_pa.replicates_needed(base, other, rng, alpha=ALPHA_USED)
            target = power_common.POWER_TARGETS[0]      # 0.80
            print(f"{label:>46} {n_seeds:>6} "
                  f"{ind_pa.fmt_r(need_mei[target], ind_pa.MAX_REPLICATES):>10} "
                  f"{ind_pa.fmt_r(need_obs[target], ind_pa.MAX_REPLICATES):>14}")
else:
    print("skipped (RUN_HEAVY=False): needs notebooks/induction/results")

### Does the classifier hold up on data shaped like the study's?

**What this is.** A calibration check on the self-test's INPUTS, prompted by a
review finding. `synth` used to draw all `n_seeds x N_HARM` marks i.i.d. from a
single rate. But `paired_diff_ci`'s own docstring says the opposite of the real
data -- harmonics inside a replicate are strata of differing difficulty, never
exchangeable draws -- and section 2 reports `design_effect` on the real
induction results for exactly that reason. The self-test was therefore printing
`PASSED` on data the study says it does not have.

**What clustering does to the verdicts.** Give each replicate an arm-specific
logit offset shared by its 9 harmonics (`CLUSTER_SD`, calibrated to a median
`design_effect` of about 3.0) and a TRUE NULL changes character: the unpaired
CMH denominator omits the within-replicate covariance term -- which is precisely
what `design_effect` measures -- so it is anticonservative, and DECIDED, every
one of them a false rejection, goes from about 5% to about 20% while EQUIVALENT
collapses. The cell below measures it; the numbers it prints are the evidence,
not this paragraph.

**Why the clustered case is REPORTED and never asserted.** The self-test exists
to pin the classifier's pure decision logic, and its i.i.d. assertions are
deliberately KEPT for that job: they are about `classify`, which is correct, and
they are stable. The clustered result is about the classifier's INPUTS instead,
and under clustering a single draw's verdict is not stable -- asserting
EQUIVALENT there was measured failing 38 times in 60 at deff 3.19. An assertion
would buy a flaky notebook and no information. One draw is an illustration; the
distribution below is the evidence.

**This is not a new finding.** It is the same defect PR #12 found in
`multiplicity_sim` -- a simulation that validated a method against i.i.d. data
the study does not collect. The classifier's calibration under clustering is
tracked separately and is NOT fixed here. What changes here is that the notebook
now shows the gap instead of printing `PASSED` over it.

In [ ]:
"""Verdict distribution under clustering: a calibration check, not a study statistic."""
import collections

import numpy as np


def verdict_distribution(cluster_sd, n_sim=60, rate=0.5, n_seeds=40, mei=0.15,
                         alpha=0.05, seed=20260904) -> dict:
    """Tally the classifier's verdicts over repeated TRUE-NULL contrasts.

    Both arms are drawn from the SAME `rate`, so every contrast is a true null
    and every DECIDED is a FALSE REJECTION; the i.i.d. column should therefore
    sit near `alpha`. This is a calibration check on the classifier's inputs,
    NOT a statistic the study reports -- nothing here reaches the paper, and
    the real-data cell above does not consume it.

    Parameters
    ----------
    cluster_sd : float
        Passed to `synth`. ``0.0`` draws harmonics i.i.d.; larger values give
        each replicate an arm-specific logit offset. `CLUSTER_SD` is the value
        calibrated against `paired.design_effect`.
    n_sim : int, default 60
        Contrasts to simulate. ONE generator is threaded through all of them,
        so the draws are independent across simulations.
    rate : float, default 0.5
        Success rate for BOTH arms -- what makes every contrast a true null.
    n_seeds : int, default 40
        Replicates per arm.
    mei : float, default 0.15
        Minimum effect of interest handed to `classify`.
    alpha : float, default 0.05
        Significance threshold. The interval is taken at ``2 * alpha``, exactly
        as `posterior_state` does, so this and the self-test stay comparable.
    seed : int, default 20260904
        Generator seed, fixed so the printed table is reproducible.

    Returns
    -------
    dict
        ``verdicts`` -- a ``collections.Counter`` over DECIDED / EQUIVALENT /
        UNDECIDED, summing to `n_sim`; ``median_deff`` -- the median
        `paired.design_effect` over the draws where it returned a value; and
        ``n_sim``. `paired.design_effect` returns ``None`` for a degenerate
        stratum, and those draws are SKIPPED rather than counted as 1.0, which
        would drag the median toward "no clustering" precisely when the
        measurement failed.

    Raises
    ------
    ValueError
        If `design_effect` returned ``None`` for every draw, so there is no
        median to report. Returning a nan instead would print as a number.
    """
    gen = np.random.default_rng(seed)
    verdicts: collections.Counter = collections.Counter()
    deffs: list[float] = []
    for _ in range(n_sim):
        a, seed_idx = synth(rate, n_seeds, gen, cluster_sd=cluster_sd)
        b, _ = synth(rate, n_seeds, gen, cluster_sd=cluster_sd)
        # The same pipeline `posterior_state` runs, so the self-test's single
        # draw and this distribution are measuring the same thing.
        p = paired.cmh_unpaired_p(a, b, seed_idx)
        ci = paired_diff_ci(a, b, seed_idx, alpha=2 * alpha, seed=0)
        verdicts[classify(p, ci["lo"], ci["hi"], mei, n_seeds, alpha)] += 1
        deff = paired.design_effect(a, b, seed_idx)
        if deff is not None:
            deffs.append(deff)
    if not deffs:
        raise ValueError(
            f"verdict_distribution: design_effect returned None for all {n_sim} "
            f"draws at cluster_sd={cluster_sd}, so the clustering this cell "
            "claims to measure cannot be verified"
        )
    return {"verdicts": verdicts, "median_deff": float(np.median(deffs)),
            "n_sim": n_sim}


_VD_ALPHA = 0.05
_vd_iid = verdict_distribution(cluster_sd=0.0, alpha=_VD_ALPHA)
_vd_clu = verdict_distribution(cluster_sd=CLUSTER_SD, alpha=_VD_ALPHA)

print(f"Verdict distribution over {_vd_iid['n_sim']} TRUE-NULL contrasts "
      f"(0.50 vs 0.50, R=40, MEI=0.15, alpha={_VD_ALPHA}).")
print("Both columns are true nulls, so every DECIDED is a FALSE REJECTION.\n")
print(f"{'':>26s} {'i.i.d. harmonics':>18s} {'clustered':>18s}")
print("-" * 64)
print(f"{'synth cluster_sd':>26s} {0.0:>18.1f} {CLUSTER_SD:>18.1f}")
print(f"{'measured median deff':>26s} {_vd_iid['median_deff']:>18.2f} "
      f"{_vd_clu['median_deff']:>18.2f}")
for _vd_state in ("DECIDED", "EQUIVALENT", "UNDECIDED"):
    print(f"{_vd_state:>26s} {_vd_iid['verdicts'][_vd_state]:>18d} "
          f"{_vd_clu['verdicts'][_vd_state]:>18d}")
_vd_iid_rate = _vd_iid["verdicts"]["DECIDED"] / _vd_iid["n_sim"]
_vd_clu_rate = _vd_clu["verdicts"]["DECIDED"] / _vd_clu["n_sim"]
print(f"{'DECIDED rate':>26s} {_vd_iid_rate:>17.1%} {_vd_clu_rate:>17.1%}")

# The conclusion as computed figures, so it cannot drift from the table above.
print(f"\nOn a TRUE NULL the i.i.d. column sits at {_vd_iid_rate:.1%} against a "
      f"nominal {_VD_ALPHA:.0%}; clustering\ntakes it to {_vd_clu_rate:.1%}, "
      f"{_vd_clu_rate / _VD_ALPHA:.1f}x the nominal rate, while EQUIVALENT falls "
      f"from {_vd_iid['verdicts']['EQUIVALENT']} to "
      f"{_vd_clu['verdicts']['EQUIVALENT']}.\nThe unpaired CMH denominator "
      f"omits the within-replicate covariance term that\n"
      f"paired_analysis.design_effect measures, so it is anticonservative here. "
      f"This is a\ncalibration warning about the classifier's INPUTS -- see the "
      f"note above, and PR #12.")

del _VD_ALPHA, _vd_iid, _vd_clu, _vd_state, _vd_iid_rate, _vd_clu_rate

### What design effect can a DECIDED verdict tolerate?

**What this is.** The verdict-distribution cell above demonstrates the
MECHANISM at R=40, MEI=0.15, alpha=0.05, an operating point chosen because 60
draws are already enough to see the unpaired CMH denominator turn
anticonservative under clustering. That is not the study's own operating
point, and a false-DECIDED rate is a tail probability: it moves by orders of
magnitude with alpha, not by a constant factor, so the mechanism demo's
numbers do not transfer. The cell below re-runs the same kind of measurement
at the study's actual `run_study.N_REPLICATES` and `ALPHA_POSTERIOR`,
sweeping `synth`'s clustering knob from none up to `CLUSTER_SD` itself.

**Why no interval, and why no MEI.** `classify`'s `DECIDED` branch is
`p < alpha` alone -- `ci_lo`/`ci_hi` never enter it. So this measurement never
calls `paired_diff_ci` or its bootstrap, and MEI (which only bounds the
EQUIVALENT branch) is not a parameter of it either. Skipping the bootstrap is
also what makes thousands of draws affordable at an alpha this small, where
even the capped bootstrap above would not resolve the tail.

**The rule.** A DECIDED verdict is valid only where the classifier is
calibrated: at or below the design effect ceiling the cell below prints, no
inflation was measured; above it, the false-DECIDED rate departs from
alpha, exactly as the distribution above already showed at R=40. Compare a
contrast's own measured `design_effect` -- the same statistic section 2
reports on the real induction results -- against that printed ceiling before
trusting a DECIDED verdict on it. The number belongs to the table the cell
prints, not to this paragraph, so it cannot drift out of step with a re-run.

**The detection floor.** An admissible rung means no inflation was MEASURED
at this sample size, not that the rung is calibrated to alpha: at this alpha
a nominal rung is expected to produce well under one false DECIDED, so a
zero (or near-zero) count only bounds the true rate loosely. The cell states
that floor alongside the ceiling. A larger sample size can only lower the
ceiling, never raise it.

In [ ]:
"""False-DECIDED calibration at the study's own R and alpha, not the mechanism demo's."""
import numpy as np
from scipy.stats import binomtest

#: Draws per rung. `false_decided_rate` never calls `paired_diff_ci` (see its
#: docstring below), so a draw costs only `paired.cmh_unpaired_p` +
#: `paired.design_effect` -- measured at roughly 3,600 draws/second, which is
#: why the whole 8-rung ladder below runs in about 9 seconds.
CAL_N_SIM = 4000

#: The study's own replicate count, read from the driver rather than typed: a
#: roster edit that changes `run_study.N_REPLICATES` moves this with it.
STUDY_R = run_study.N_REPLICATES

#: `synth`'s clustering knob, swept from none up to `CLUSTER_SD` itself, so
#: the ladder brackets the calibrated value from below.
CAL_CLUSTER_SDS = (0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2, CLUSTER_SD)


def false_decided_rate(cluster_sd, n_sim=CAL_N_SIM, *, r=STUDY_R,
                       alpha=ALPHA_POSTERIOR, rate=0.5, seed=20260906,
                       gen=None) -> dict:
    """Measure the classifier's false-DECIDED rate on a true null, under clustering.

    Both arms of every draw come from the SAME `rate`, so every contrast
    simulated here is a true null and every DECIDED is a false rejection.
    `classify`'s DECIDED branch is ``p < alpha`` alone -- `ci_lo`/`ci_hi` never
    enter it -- so this measures DECIDED directly off `paired.cmh_unpaired_p`
    and never calls `paired_diff_ci` or its bootstrap. That is also why MEI is
    not a parameter here: MEI only bounds the EQUIVALENT branch, which a
    true-null false rejection never reaches. Skipping the bootstrap is what
    makes `n_sim` in the thousands affordable at an alpha this small: one
    `paired_diff_ci` call alone costs more than one whole draw does here.

    Parameters
    ----------
    cluster_sd : float
        Passed to `synth`. ``0.0`` is the i.i.d. control; `CLUSTER_SD` is the
        value calibrated against the real data's median `design_effect`.
    n_sim : int, default CAL_N_SIM
        Contrasts to simulate.
    r : int, keyword-only, default STUDY_R
        Replicates per arm -- the study's own `run_study.N_REPLICATES` unless
        overridden.
    alpha : float, keyword-only, default ALPHA_POSTERIOR
        Significance threshold DECIDED is measured against -- the all-pairs
        posterior family's alpha unless overridden.
    rate : float, keyword-only, default 0.5
        Success rate for BOTH arms -- what makes every contrast a true null.
    seed : int, keyword-only, default 20260906
        Seed for a fresh generator when `gen` is not supplied. Fixed so a
        standalone call is reproducible.
    gen : numpy.random.Generator or None, keyword-only, default None
        Drawn from in place. ``None`` builds
        ``numpy.random.default_rng(seed)``; a caller that supplies its own
        generator OWNS that stream and is responsible for threading it. The
        calibration ladder below builds ONE generator and passes it to every
        rung, so the rungs draw independent contrasts instead of sharing
        common random numbers across `cluster_sd` values.

    Returns
    -------
    dict
        ``cluster_sd``, ``r``, ``alpha``, ``n_sim``, ``decided`` (count),
        ``rate`` (``decided / n_sim``), ``median_deff`` (median
        `paired.design_effect` over the draws where it returned a value,
        skipping ``None``s the same way `verdict_distribution` does),
        ``ci_lo``/``ci_hi`` (95% Clopper-Pearson interval on `rate`, via
        `scipy.stats.binomtest`), and ``inflated`` (``ci_lo > alpha``: the
        observed count is too high to be the nominal rate).

    Raises
    ------
    ValueError
        If `design_effect` returned ``None`` for every draw, so there is no
        median to report -- the same refusal `verdict_distribution` makes,
        for the same reason: a stand-in value would print as a measurement.

    Notes
    -----
    O(`n_sim`) calls to `synth`, `paired.cmh_unpaired_p` and
    `paired.design_effect`; no bootstrap and no interval other than the
    Clopper-Pearson one on the tallied rate itself.
    """
    if gen is None:
        gen = np.random.default_rng(seed)
    decided = 0
    deffs: list[float] = []
    for _ in range(n_sim):
        a, seed_idx = synth(rate, r, gen, cluster_sd=cluster_sd)
        b, _ = synth(rate, r, gen, cluster_sd=cluster_sd)
        # DECIDED is `p < alpha` alone -- see the docstring above for why
        # `paired_diff_ci` is never called on this path.
        if paired.cmh_unpaired_p(a, b, seed_idx) < alpha:
            decided += 1
        deff = paired.design_effect(a, b, seed_idx)
        if deff is not None:
            deffs.append(deff)
    if not deffs:
        raise ValueError(
            f"false_decided_rate: design_effect returned None for all {n_sim} "
            f"draws at cluster_sd={cluster_sd}, so the clustering this rung "
            "claims to measure cannot be verified"
        )
    median_deff = float(np.median(deffs))
    ci_lo, ci_hi = binomtest(decided, n_sim).proportion_ci(
        confidence_level=0.95, method="exact")
    return {
        "cluster_sd": cluster_sd, "r": r, "alpha": alpha, "n_sim": n_sim,
        "decided": decided, "rate": decided / n_sim, "median_deff": median_deff,
        "ci_lo": ci_lo, "ci_hi": ci_hi, "inflated": ci_lo > alpha,
    }


# ONE generator threads the whole ladder -- the rungs are independent draws
# rather than sharing common random numbers across `cluster_sd` values, the
# same reason `verdict_distribution` threads a single generator through its
# own draws.
_cal_gen = np.random.default_rng(20260906)
CALIBRATION_ROWS = [false_decided_rate(sd, gen=_cal_gen) for sd in CAL_CLUSTER_SDS]

# Walk the ladder in order: the ceiling is the median_deff of the LAST rung
# BEFORE the first inflated one. If the very first rung is already inflated
# there is no design-effect range at which no inflation was measured -- refuse
# rather than print a stand-in number, the same rule the cell above follows
# for a missing median.
CALIBRATED_DEFF_CEILING = None
for _row in CALIBRATION_ROWS:
    if _row["inflated"]:
        break
    CALIBRATED_DEFF_CEILING = _row["median_deff"]
if CALIBRATED_DEFF_CEILING is None:
    raise ValueError(
        "false-DECIDED calibration: the first rung "
        f"(cluster_sd={CALIBRATION_ROWS[0]['cluster_sd']}) is already "
        f"inflated at n_sim={CAL_N_SIM}, so no design-effect range measured "
        "no inflation -- refusing to report a ceiling"
    )

_expected_false = CAL_N_SIM * ALPHA_POSTERIOR
print(f"False-DECIDED rate at R={STUDY_R} (run_study.N_REPLICATES), "
      f"alpha = power_common.ALPHA/N_TESTS = {power_common.ALPHA}/{N_TESTS} "
      f"= {ALPHA_POSTERIOR:.3e}, n_sim={CAL_N_SIM}.")
print("Both arms are drawn from the SAME rate at every cluster_sd, so every "
      "DECIDED below is a false rejection.\n")

print(f"{'cluster_sd':>10} {'median deff':>12} {'decided/n_sim':>16} "
      f"{'rate':>10} {'95% CI (Clopper-Pearson)':>28}  verdict")
print("-" * 96)
for _row in CALIBRATION_ROWS:
    _count_txt = f"{_row['decided']}/{_row['n_sim']}"
    _ci_txt = f"[{_row['ci_lo']:.2e}, {_row['ci_hi']:.2e}]"
    _verdict = "INFLATED" if _row["inflated"] else "no measured inflation"
    print(f"{_row['cluster_sd']:>10.1f} {_row['median_deff']:>12.3f} "
          f"{_count_txt:>16} {_row['rate']:>10.4%} {_ci_txt:>28}  {_verdict}")

_iid_row = CALIBRATION_ROWS[0]
_iid_count_txt = f"{_iid_row['decided']}/{_iid_row['n_sim']}"

# The i.i.d. rung's own CP upper bound (above) answers "how high could THIS
# rung's true rate be, given what it read" -- a DIFFERENT question from "how
# small an observed count would this sweep even notice". The latter is the
# smallest k whose OWN CP lower bound clears alpha, i.e. the smallest count
# `inflated` would ever flag; walk k up from 0 rather than substitute the
# i.i.d. rung's bound for it, which would answer the wrong question.
_k = 0
while (binomtest(_k, CAL_N_SIM).proportion_ci(confidence_level=0.95, method="exact").low
       <= ALPHA_POSTERIOR):
    _k += 1
_detect_rate = _k / CAL_N_SIM

print(f"\nDetection floor: at this alpha a NOMINAL rung is EXPECTED to produce "
      f"only {_expected_false:.3g} false DECIDEDs out of {CAL_N_SIM} draws. The "
      f"i.i.d. rung above, reading {_iid_count_txt}, only bounds its OWN true "
      f"rate up to about its 95% upper bound ({_iid_row['ci_hi']:.2e}, roughly "
      f"{_iid_row['ci_hi'] / ALPHA_POSTERIOR:.0f}x alpha). Separately, the "
      f"smallest OBSERVED count this sweep could DETECT as inflated is "
      f"{_k}/{CAL_N_SIM} = {_detect_rate:.2e}, roughly "
      f"{_detect_rate / ALPHA_POSTERIOR:.1f}x alpha -- below that count, "
      "'no measured inflation' means exactly that: nothing was measured, not "
      "that alpha was confirmed. An admissible rung means NO MEASURED "
      "INFLATION at this n_sim, not 'calibrated to alpha'. A larger n_sim can "
      "only LOWER this ceiling, never raise it.")

print(f"\nCALIBRATED_DEFF_CEILING = {CALIBRATED_DEFF_CEILING:.2f}: no rung at "
      "or below this design effect showed measured inflation.")
_study_row = CALIBRATION_ROWS[-1]
print(f"At the study-shaped rung (cluster_sd={_study_row['cluster_sd']}, "
      f"matching CLUSTER_SD): {_study_row['decided']}/{_study_row['n_sim']} "
      f"DECIDED, rate {_study_row['rate']:.4%} = "
      f"{_study_row['rate'] / ALPHA_POSTERIOR:.1f}x alpha.")

print("\npaired_analysis.design_effect is the same statistic section 2 "
      "reports on the real induction results -- that is where a reader "
      "compares this ceiling against their own measurement.")

del (_cal_gen, _row, _expected_false, _count_txt, _ci_txt, _verdict,
     _iid_row, _iid_count_txt, _k, _detect_rate, _study_row)

## Section 8 -- score-level flip rate

**What this is.** The study's direct measurement of cross-**process**
generation nondeterminism. 200 Mathlib-only measurable cells (any
``.lake/packages/`` path -- Batteries, formerly named Std, and every
other Lake dependency -- excluded) of the `nemotron-3-nano-4b` deduction
lane were re-generated on a fresh box, and
both legs were graded by **today's** verifier, so verifier drift cannot leak
into the comparison (a separate `verifier_drift_stats` isolates that
instead). The estimator is a McNemar-style 2x2 of
re-verified-original vs rerun pass@1, reporting `b + c` discordant cells, the
flip rate, an **exact Clopper-Pearson** interval on it, and the implied
normal-approximation SE on pass@1.

That SE and the CP interval both assume independent cells. Several cells in a
200-cell sample can share a theorem, and a theorem-level effect would
correlate their flips -- so `flip_stats` carries the caveat *inside the JSON
report*, and this notebook prints it verbatim rather than quoting the number
alone.

**Inputs.** The archived report
`notebooks/deduction/results/runs/flip_nemotron-3-nano-4b/flip_report.json`,
streamed from S3. The originals leg
(`originals_rerun/`) is **never** touched and never leaves the machine it was
graded on, by that script's own hard rule.

**What was ported.** The pure estimators only:
`clopper_pearson_interval` (with `_binom_cdf`/`_bisect_decreasing`),
`flip_stats`, `verifier_drift_stats`, `is_pass`, `group_rows_by_cell`,
`surviving_verdict`, `measurable_cell_keys`, `select_sample_keys`.
Measurability itself is **not** ported: `measurable_cell_keys` now derives it
from the live `ded_pa.UNMEASURABLE_VERDICTS` and the earliest-surviving-attempt
rule of `ded_pa.grade_verdicts`, instead of the positive verdict whitelist it
used to carry -- two tables that had to stay exact complements while sitting in
different files, free to drift apart.
Everything else in that 1748-line script is orchestration (EC2, spooling,
Lean verification, stage plumbing) -- see the closing section.

In [ ]:
"""The pure flip-rate estimators."""
import math
import random
from collections.abc import Iterable, Mapping, Sequence

from smolbench.deduction.lean import runner

#: Theorem ``file_path`` marker for a Lake dependency (not Mathlib)
#: theorem. ANY ``.lake/packages/`` path is a vendored dependency --
#: Batteries (formerly named ``std``), aesop, plausible, and every other
#: Lake package -- so this matches the directory, not one package name;
#: matching only ``std/`` would silently reclassify every OTHER
#: dependency's theorems as Mathlib once mathlib4 renamed its ``std``
#: dependency to ``batteries``.
_DEPENDENCY_PACKAGE_MARKER = ".lake/packages/"

#: The generation-time sentinel for a row that was never graded. `power_analysis`
#: keeps it OUT of ``UNMEASURABLE_VERDICTS`` on purpose, so its loaders can refuse
#: it loudly (`reject_unverified_verdicts`) instead of dropping it silently. The
#: job here is the other one -- choosing a population to SAMPLE, not loading one to
#: score -- and a cell whose surviving attempt is ungraded has no measurement to
#: sample: `is_pass` raises on it by design. Excluding it here is what makes that
#: docstring's cross-reference ("Filter ungraded rows out before calling is_pass
#: (see measurable_cell_keys)") true.
UNGRADED_VERDICT = "unverified"


def group_rows_by_cell(rows: Iterable[dict]) -> dict[tuple, list[dict]]:
    """Group rows by cell key, keeping ALL of a cell's rows in file order.

    File order is chronological order, which is the order
    `ded_pa.grade_verdicts` walks its verdicts in. Keeping the whole group --
    rather than collapsing each cell to its first row -- is what lets the
    grader's rule be applied downstream instead of a first-row approximation
    of it: a cell whose first attempt raised and whose retry succeeded is only
    visible as a group.
    """
    by_key: dict[tuple, list[dict]] = {}
    for row in rows:
        if row.get("kind") != "cell":
            continue
        key = runner._row_key(
            row.get("model", ""), row.get("theorem_id", ""),
            int(row.get("k", -1)), row.get("rung", ""),
            int(row.get("replicate_idx", -1)),
        )
        by_key.setdefault(key, []).append(row)
    return by_key


def surviving_verdict(verdicts: Iterable[str | None]) -> str | None:
    """Return the first verdict that survives `ded_pa.UNMEASURABLE_VERDICTS`.

    ``None`` when nothing survives. The skip set is read off `ded_pa` at call
    time and never copied into a local literal here: a copy is exactly the
    drift defect this helper replaces, relocated.

    This sits NEXT TO `ded_pa.grade_verdicts` rather than replacing it. The
    grader collapses its answer to 1/0/``None``, which loses the IDENTITY of
    the verdict it acted on, and section 8 needs that identity: it has to tell
    the ungraded sentinel apart from a real failure, and to the grader both are
    a measurement scoring 0 -- only one of them is one.

    A row carrying no ``verdict`` field yields ``None`` too, indistinguishable
    from "nothing survived". `measurable_cell_keys` treats the two the same on
    purpose: neither is a measurement it could sample.
    """
    for verdict in verdicts:
        if verdict not in ded_pa.UNMEASURABLE_VERDICTS:
            return verdict
    return None


# The grade of the surviving verdict must be exactly what the live grader
# returns, for every shape of input. If power_analysis' row rule changes, this
# fails HERE rather than leaving section 8 sampling a population the study's
# own loaders would not recognise.
#
# Guarded on the GRADER'S PRESENCE, never on the assertion's outcome: wherever
# a grader is bound -- cell 2 binds `ded_pa` for every real run of this
# notebook, and section 8's tests bind it too -- disagreement still raises.
# The one context without one is
# tests/deduction/test_postcutoff_docs.py::test_dependency_filter_covers_every_lake_package,
# which execs this cell into an EMPTY namespace to reach `is_mathlib_cell`;
# there is no live rule there to pin against, so there is nothing to check.
if "ded_pa" in globals():
    for _verdicts in ([], ["exception"], ["exception", "replay_failed"],
                      ["exception", "success"], ["success", "failure"],
                      ["replay_failed", "incomplete"]):
        _survivor = surviving_verdict(_verdicts)
        assert ded_pa.grade_verdicts(_verdicts) == (
            None if _survivor is None else int(_survivor == "success")), _verdicts
    del _verdicts, _survivor


def is_mathlib_cell(row: dict) -> bool:
    """Check that a cell's theorem is vendored from Mathlib, not a Lake dependency.

    A missing ``file_path`` is not evidence of being a dependency theorem,
    so it is treated as Mathlib.
    """
    return not str(row.get("file_path") or "").startswith(_DEPENDENCY_PACKAGE_MARKER)


def measurable_cell_keys(rows: Iterable[dict]) -> list[tuple]:
    """Sorted Mathlib-only cell keys whose surviving verdict is a measurement.

    Membership is the LIVE grader's rule, not a local table: a cell is in the
    population iff `surviving_verdict` finds an attempt outside
    `ded_pa.UNMEASURABLE_VERDICTS` and that attempt was actually graded (not
    `UNGRADED_VERDICT`). Ranking a hand-written positive whitelist ahead of the
    grader lost a real case -- a cell whose first attempt raised
    (``"exception"``) and whose retry succeeded is measurable, and the grader
    scores it 1, but collapsing the cell to its FIRST row left only the
    exception to test against the whitelist and the cell fell out of the
    population entirely.

    The ascending sort is what makes `select_sample_keys` reproducible
    independent of the rows' on-disk order.
    """
    keys: list[tuple] = []
    for key, group in group_rows_by_cell(rows).items():
        verdict = surviving_verdict(row.get("verdict") for row in group)
        if verdict is None or verdict == UNGRADED_VERDICT:
            continue
        # `file_path` identifies the THEOREM, not the attempt, so which attempt
        # survived cannot change this answer; group[0] is representative.
        if not is_mathlib_cell(group[0]):
            continue
        keys.append(key)
    return sorted(keys)


def select_sample_keys(measurable: Sequence[tuple], n: int, seed: int) -> list[tuple]:
    """Draw a reproducible n-cell sample from an ALREADY-SORTED population."""
    return random.Random(seed).sample(list(measurable), n)


def _binom_cdf(k: int, n: int, p: float) -> float:
    """``P(X <= k)`` for ``X ~ Binomial(n, p)``, in pure stdlib."""
    if p <= 0.0:
        return 1.0
    if p >= 1.0:
        return 1.0 if k >= n else 0.0
    return sum(math.comb(n, i) * p ** i * (1 - p) ** (n - i) for i in range(k + 1))


def _bisect_decreasing(f, lo: float, hi: float, iters: int = 100) -> float:
    """Root of a MONOTONE DECREASING `f` on ``[lo, hi]``, by bisection."""
    for _ in range(iters):
        mid = (lo + hi) / 2
        if f(mid) > 0:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2


def clopper_pearson_interval(k: int, n: int, alpha: float = 0.05) -> tuple[float, float]:
    """Exact binomial (Clopper-Pearson) interval, by bisection on the CDF.

    ``lower == 0.0`` iff ``k == 0`` and ``upper == 1.0`` iff ``k == n``:
    the standard boundary convention, since there is no informative bound
    to solve for at either extreme. Bisection (not ``scipy.stats.beta.ppf``)
    keeps the numbers identical without depending on scipy's implementation.
    """
    if n <= 0:
        raise ValueError(f"clopper_pearson_interval: n must be positive, got {n}")
    if not 0 <= k <= n:
        raise ValueError(f"clopper_pearson_interval: k must be in [0, {n}], got {k}")
    lower = (0.0 if k == 0 else
             _bisect_decreasing(lambda p: _binom_cdf(k - 1, n, p) - (1 - alpha / 2), 0.0, 1.0))
    upper = (1.0 if k == n else
             _bisect_decreasing(lambda p: _binom_cdf(k, n, p) - alpha / 2, 0.0, 1.0))
    return lower, upper


def is_pass(verdict: str) -> bool:
    """``True`` iff `verdict` is exactly ``"success"``.

    Raises on the generation-time sentinel ``"unverified"``, deliberately
    rather than returning False: scoring "never measured" as "measured and
    lost" biases every paired b/c statistic downward, invisibly.
    """
    if verdict == "unverified":
        raise ValueError(
            'is_pass: verdict is "unverified" -- the generation-time sentinel '
            "for an ungraded row, not a graded outcome. Filter ungraded rows "
            "out before calling is_pass (see measurable_cell_keys)."
        )
    return verdict == "success"


PASS_AT_1_SE_CAVEAT = (
    "Normal-approximation SE (and the Clopper-Pearson CI) assume "
    "independent cells. Several cells in this sample can share a "
    "theorem (different rungs/replicates of it), which this "
    "estimate does not account for -- see flip_stats' docstring "
    "Notes. Treat as a rough, likely-too-narrow bound, not exact."
)


def flip_stats(pairs: Mapping[tuple, tuple[str, str]]) -> dict:
    """McNemar-style flip statistics: re-verified original vs rerun verdict.

    `pairs` maps a cell key to the two verdict TEXTS, never pre-computed
    booleans, so a caller cannot silently apply a different pass rule to
    the two legs: this function applies `is_pass` to both itself.
    """
    n = len(pairs)
    a = b = c = d = 0
    flipped_keys: list[tuple] = []
    for key, (orig_verdict, rerun_verdict) in pairs.items():
        orig, rerun = is_pass(orig_verdict), is_pass(rerun_verdict)
        if orig and rerun:
            a += 1
        elif orig and not rerun:
            b += 1
            flipped_keys.append(key)
        elif not orig and rerun:
            c += 1
            flipped_keys.append(key)
        else:
            d += 1
    discordant = b + c
    flip_rate = discordant / n if n else 0.0
    ci_lo, ci_hi = clopper_pearson_interval(discordant, n) if n else (0.0, 0.0)
    se = math.sqrt(flip_rate * (1 - flip_rate) / n) if n else 0.0
    return {
        "n": n,
        "a_both_pass": a,
        "b_orig_pass_rerun_fail": b,
        "c_orig_fail_rerun_pass": c,
        "d_both_fail": d,
        "discordant": discordant,
        "flip_rate": flip_rate,
        "flip_rate_ci95": [ci_lo, ci_hi],
        "pass_at_1_se": se,
        "pass_at_1_se_caveat": PASS_AT_1_SE_CAVEAT,
        "flipped_keys": [list(k) for k in flipped_keys],
    }


def verifier_drift_stats(pairs: Mapping[tuple, tuple[str, str]]) -> dict:
    """Agreement of a fresh reverification against the study's stored verdict.

    Compares the two verdict STRINGS for equality rather than collapsing
    them to pass/fail: drift between e.g. ``lean_error`` and ``incomplete``
    is informative even though neither is a success.
    """
    n = len(pairs)
    agree = 0
    disagreements: list[dict] = []
    for key, (study_verdict, reverified_verdict) in pairs.items():
        if study_verdict == reverified_verdict:
            agree += 1
        else:
            disagreements.append({"key": list(key), "study_verdict": study_verdict,
                                  "reverified_verdict": reverified_verdict})
    return {"n": n, "agree": agree, "agreement_rate": agree / n if n else 0.0,
            "disagreements": disagreements}


print("ported estimators:", ", ".join(sorted(
    ("clopper_pearson_interval", "flip_stats", "verifier_drift_stats", "is_pass",
     "group_rows_by_cell", "surviving_verdict", "measurable_cell_keys",
     "select_sample_keys"))))

In [ ]:
"""Re-render this section's flip table from the archive, and ASSERT it matches."""
FLIP_REPORT = ("notebooks/deduction/results/runs/flip_nemotron-3-nano-4b/"
               "flip_report.json")
SAMPLE_MANIFEST = ("notebooks/deduction/results/runs/flip_nemotron-3-nano-4b/"
                   "sample_manifest.json")

report = archive.json(FLIP_REPORT)
manifest = archive.json(SAMPLE_MANIFEST)
stored = report["flip_stats"]

# Rebuild the pairing the stored 2x2 describes, one DISTINCT key per cell, and
# push it back through the ported flip_stats. This exercises is_pass on real
# verdict strings rather than trusting the stored arithmetic.
pairs = {}
for i in range(stored["a_both_pass"]):
    pairs[("rebuilt", "a", i, "-", 0)] = ("success", "success")
for i in range(stored["b_orig_pass_rerun_fail"]):
    pairs[("rebuilt", "b", i, "-", 0)] = ("success", "lean_error")
for i in range(stored["c_orig_fail_rerun_pass"]):
    pairs[("rebuilt", "c", i, "-", 0)] = ("lean_error", "success")
for i in range(stored["d_both_fail"]):
    pairs[("rebuilt", "d", i, "-", 0)] = ("lean_error", "lean_error")
assert len(pairs) == stored["n"], (len(pairs), stored["n"])
rendered = flip_stats(pairs)

print(f"lane {report['model']}   study run {report['study_run']}   "
      f"flip run {report['flip_run']}")
print(f"sample: {report['n_paired']}/{report['n_requested']} paired, drawn from "
      f"{report['sample_n_measurable_mathlib_population']} measurable Mathlib "
      f"cells (whitelist sha256 {report['sample_whitelist_sha256'][:16]})")
print(f"manifest agrees on the population: "
      f"{manifest.get('n_measurable_mathlib_population')}")
print(f"generated {report['generated_at_utc']}")

print(f"\n2x2 (re-verified original x rerun), n = {rendered['n']}")
print(f"{'':>18} {'rerun pass':>12} {'rerun fail':>12}")
print(f"{'orig pass':>18} {rendered['a_both_pass']:>12} "
      f"{rendered['b_orig_pass_rerun_fail']:>12}")
print(f"{'orig fail':>18} {rendered['c_orig_fail_rerun_pass']:>12} "
      f"{rendered['d_both_fail']:>12}")
print(f"\ndiscordant (b + c) : {rendered['discordant']}")
print(f"flip rate          : {rendered['flip_rate']:.4f}")
print(f"95% Clopper-Pearson: [{rendered['flip_rate_ci95'][0]:.6f}, "
      f"{rendered['flip_rate_ci95'][1]:.6f}]")
print(f"implied pass@1 SE  : {rendered['pass_at_1_se']:.6f}")
print(f"\nCAVEAT: {rendered['pass_at_1_se_caveat']}")

drift = report["verifier_drift"]
print(f"\nverifier drift (same text, today's verifier vs the study's stored "
      f"verdict): {drift['agree']}/{drift['n']} = {drift['agreement_rate']:.4f}")

# The asserts. Every expected value is READ from the archived JSON.
# a/b/c/d are the counts the pairing above was RECONSTRUCTED from, so they
# check the reconstruction; the statistics DERIVED from them -- discordant,
# flip_rate, the Clopper-Pearson interval, the SE, and the caveat string --
# are what the ported estimator actually re-computes here.
for field in ("n", "a_both_pass", "b_orig_pass_rerun_fail",
              "c_orig_fail_rerun_pass", "d_both_fail", "discordant",
              "flip_rate", "pass_at_1_se", "pass_at_1_se_caveat"):
    assert rendered[field] == stored[field], (field, rendered[field], stored[field])
assert rendered["flip_rate_ci95"] == stored["flip_rate_ci95"], (
    rendered["flip_rate_ci95"], stored["flip_rate_ci95"])
# The drift table is a pure recount, so it re-renders exactly too.
drift_pairs = {("rebuilt", "agree", i, "-", 0): ("lean_error", "lean_error")
               for i in range(drift["agree"])}
drift_pairs.update({("rebuilt", "disagree", i, "-", 0):
                    (d["study_verdict"], d["reverified_verdict"])
                    for i, d in enumerate(drift["disagreements"])})
rendered_drift = verifier_drift_stats(drift_pairs)
for field in ("n", "agree", "agreement_rate"):
    assert rendered_drift[field] == drift[field], (field, rendered_drift[field], drift[field])

print("\nASSERT OK: discordant, flip rate, Clopper-Pearson interval, pass@1 SE "
      "and the\n           caveat string all re-derive to the archived record "
      f"({FLIP_REPORT});\n           the 2x2 counts the pairing was rebuilt "
      "from check out too.")

## Section 9 -- the free flip bound

**What this is.** A zero-cost bound on section 8's flip rate. The 2026-08-15
resampling bug left a handful of deduction cells with **more than one
surviving generation attempt** -- attempts that reached the model and
returned, drawn by different serving processes. Those are already-collected
paired draws of the same cell across processes, so they bound the flip rate
for free. Both attempts of every pair were graded with the same (today's)
verifier, so verifier identity cancels within a pair.

> **The caveat that must ride every use of this number.** The sample is
> **selected on cell outcome**: a cell was re-drawn precisely because its
> first attempt looked empty or failed. So this is not an unbiased estimate.
> Because the sample conditions on the first draw, regression to the mean on
> the second argues the bound *over*-estimates the population flip rate --
> a direction that is **reasoned, not measured**. It is a sanity check on
> section 8's design, **never the headline**. Note in the numbers below that
> all 74 pairs have an empty first attempt and none has two non-empty
> attempts: that is the selection, visible.

**Inputs.** The archived report
`notebooks/deduction/results/flip_free_bound_2026-08-18.json`, streamed from
S3. Everything below is recomputed from its per-pair records
(`report["lanes"][lane]["pairs"]`), not read off its summary.

**The interval is recomputed with section 8's `clopper_pearson_interval`,**
not with the archived script's own `exact_binom_ci`. That function integrated
the beta density on a 200,000-point grid to avoid a scipy dependency; the
bisection estimator is exact to machine precision, and using one interval
estimator across both sections is worth more than keeping a second
implementation alive. The two agree to the 4 decimals the record stores.

In [ ]:
"""Recompute this section's bound from the archived per-pair records."""
FREE_BOUND = "notebooks/deduction/results/flip_free_bound_2026-08-18.json"
free = archive.json(FREE_BOUND)
stored_summary = free["summary"]

# Recompute from the pairs, never from the summary.
all_pairs = [p for lane in free["lanes"].values() for p in lane["pairs"]]
n = len(all_pairs)


def _flip(pair, last: bool = False) -> bool:
    """Re-derive a pair's flip from its VERDICTS, via section 8's pass rule."""
    passes = [is_pass(v) for v in pair["verdicts"]]
    return passes[0] != (passes[-1] if last else passes[1])


# Re-derive rather than re-sum the stored boolean, then pin the two together:
# the stored flag is a summary of these verdicts and must agree with them.
assert all(_flip(p) == p["flip_first_vs_second"] for p in all_pairs)
assert all(_flip(p, last=True) == p["flip_first_vs_last"] for p in all_pairs)
flips = sum(1 for p in all_pairs if _flip(p))
flips_last = sum(1 for p in all_pairs if _flip(p, last=True))
identical = sum(1 for p in all_pairs if p["identical_text"])
dependency_pairs = sum(1 for p in all_pairs if p["is_std"])
first_empty = sum(1 for p in all_pairs if p["first_empty"])
two_nonempty = sum(1 for p in all_pairs if p["n_nonempty_attempts"] >= 2)

mathlib = [p for p in all_pairs if not p["is_std"]]
n_math = len(mathlib)
flips_math = sum(1 for p in mathlib if _flip(p))

rate = flips / n
lo, hi = clopper_pearson_interval(flips, n)
lo_m, hi_m = clopper_pearson_interval(flips_math, n_math)

print("per-lane pairs (cells with >= 2 surviving attempts):")
for lane, rec_lane in free["lanes"].items():
    lane_pairs = rec_lane["pairs"]
    lane_flips = sum(1 for p in lane_pairs if _flip(p))
    print(f"  {lane:>16}: {len(lane_pairs):>3} pairs "
          f"(inventory expected {rec_lane['inventory_expected']}), "
          f"{lane_flips} flip(s)")

print(f"\nALL pairs        : {flips}/{n} = {rate:.4f}   "
      f"95% CP [{lo:.4f}, {hi:.4f}]")
print(f"Mathlib-only     : {flips_math}/{n_math} = {flips_math / n_math:.4f}   "
      f"95% CP [{lo_m:.4f}, {hi_m:.4f}]   (dependency pairs excluded: {dependency_pairs})")
print(f"first-vs-LAST    : {flips_last}/{n}")
print(f"identical text   : {identical}/{n}")
print(f"first attempt empty          : {first_empty}/{n}")
print(f"pairs with two non-empty text: {two_nonempty}/{n}")

print(f"\nCAVEAT (carried in the record itself): {free['caveat']}")
print("This bound is a sanity check on section 8's design. It is NEVER the "
      "headline:\nthe headline flip rate is section 8's "
      f"{archive.json(FLIP_REPORT)['flip_stats']['flip_rate']:.4f} on a "
      "randomly drawn sample.")

# Asserts against the stored headline -- all expected values READ from the JSON.
assert n == stored_summary["n_pairs"], (n, stored_summary["n_pairs"])
assert flips == stored_summary["flips_first_vs_second"], (
    flips, stored_summary["flips_first_vs_second"])
assert round(rate, 4) == stored_summary["flip_rate"], (
    round(rate, 4), stored_summary["flip_rate"])
assert [round(lo, 4), round(hi, 4)] == stored_summary["ci95"], (
    [round(lo, 4), round(hi, 4)], stored_summary["ci95"])
assert flips_last == stored_summary["flips_first_vs_last"]
assert identical == stored_summary["identical_text_pairs"]
assert dependency_pairs == stored_summary["std_pairs"]
assert first_empty == stored_summary["pairs_first_attempt_empty"]
assert two_nonempty == stored_summary["pairs_with_two_nonempty_attempts"]
# The Mathlib subset: the record stores the COUNTS but no interval, so the
# counts are asserted and the interval above is reported as derived.
assert flips_math == stored_summary["flips_mathlib_only"], (
    flips_math, stored_summary["flips_mathlib_only"])
assert n_math == stored_summary["n_mathlib_pairs"], (n_math, stored_summary["n_mathlib_pairs"])

print(f"\nASSERT OK: recomputed headline {flips}/{n} and its CI equal the "
      f"archived record ({FREE_BOUND});")
print(f"           Mathlib-only subset {flips_math}/{n_math} equals it too "
      "(its CI is derived here, not stored).")

## Scope of the ports

Sections 7-9 port **statistics** only. Orchestration, I/O, and collection-time
gates from the three archived scripts are deliberately not reproduced here.